# DSD-2021 Project (CIFAR-10)
---
## All layer inference check
- This is a python script to help you check if your RTL impelementation for all the layers are correct

## Usage
- Run all the cells to check if your HW works


In [1]:
from utils.layers_cifar10 import *
from utils.bit_operation import *
from utils.setup_cifar10 import *
from utils.scale_uart import *
from utils.board import *
import time
import numpy as np
import time

### Load dataset for image generate

In [2]:
label_list = ["Airplane", "Automobile", "Bird", "Cat", "Deer", "Dog", "Frog", \
              "Horse", "Ship", "Truck"]
# TEST ORIGIN
X_test_origin, _ = load_CIFAR10_test()
# TEST SET
X_test, y_test = load_CIFAR10_test()

# Data Pre-processing
m = [0.4935, 0.4834, 0.4472]
std = [0.2476, 0.2626, 0.2626]
# Only pre-process the test dataset
X_test = np.reshape(X_test, (X_test.shape[0], 3, 32, 32))
for i in range(3):
    X_test[:,i,:,:] = (X_test[:,i,:,:]-m[i])/std[i]

### Simulation dataset for our 8-bit MAC unit

In [3]:
X_test_ = np.load("./data/cifar10_dataset_quan/images_100.npy")

## Load network parameter
---

In [4]:
# Load quantized network param
conv1_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv1_weight_quan.npy")
conv1_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv1_bias_quan.npy")
conv2_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv2_weight_quan.npy")
conv2_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv2_bias_quan.npy")
conv3_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv3_weight_quan.npy")
conv3_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv3_bias_quan.npy")
conv4_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv4_weight_quan.npy")
conv4_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv4_bias_quan.npy")
conv5_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv5_weight_quan.npy")
conv5_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv5_bias_quan.npy")
conv6_w_ = np.load("./data/cifar10_network_quan_param/cifar10_conv6_weight_quan.npy")
conv6_b_ = np.load("./data/cifar10_network_quan_param/cifar10_conv6_bias_quan.npy")
fc1_w_   = np.load("./data/cifar10_network_quan_param/cifar10_fc1_weight_quan.npy")
fc1_b_   = np.load("./data/cifar10_network_quan_param/cifar10_fc1_bias_quan.npy")
fc2_w_   = np.load("./data/cifar10_network_quan_param/cifar10_fc2_weight_quan.npy")
fc2_b_   = np.load("./data/cifar10_network_quan_param/cifar10_fc2_bias_quan.npy")
fc3_w_   = np.load("./data/cifar10_network_quan_param/cifar10_fc3_weight_quan.npy")
fc3_b_   = np.load("./data/cifar10_network_quan_param/cifar10_fc3_bias_quan.npy")

## Test for accuracy  
---
Do inference

### Board connection

In [5]:
port_list = get_port_list()
SU = get_scale_uart(port_list)

Current OS: Windows
['COM1', 'COM5']
COM1 port cannot be connected.
COM5 port connected!


### Setting the VDMA

In [6]:
## DO NOT CHANGE 
## IT IS VDMA AND EACH MODULE'S BASE ADDRESS FOR CONTROL APB + AXI
##### PARAMETER INFORMATION
VDMA0_BASE_ADDR= 0x0c00_0000
VDMA1_BASE_ADDR= 0x0c10_0000
VDMA2_BASE_ADDR= 0x0c20_0000

FC_BASE_ADDR   = 0x0d00_0000
CONV_BASE_ADDR = 0x0d10_0000
POOL_BASE_ADDR = 0x0d20_0000

### FIXED FOR OUR NETWORK
OP_SIZE                        = 4
ADDR_SIZE                      = 28
DATA_SIZE                      = 32

Image address memory map  
---
Addresss range: 0x0000_0000 ~ 0x01FF_FFFF    
Size: 32768 KB

In [7]:
start = time.time()
SU.su_write_data(0x0000_0000, 3)
data = SU.su_read_data(0x0000_0000)
SU.su_set_image({'BASE_ADDR': 0x0000_0000}, "./data/cifar10_dataset_quan/images_100.npy")
print("image set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

image set done
	Total time: 31.04 sec


Conv1 memory map
---
Convolution 1  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0200_0000 ~ 0x020F_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0210_0000 ~ 0x021F_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0600_0000 ~ 0x060F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [8]:
print("conv1 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x0200_0000}, "./data/cifar10_network_quan_param/cifar10_conv1_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x0210_0000}, "./data/cifar10_network_quan_param/cifar10_conv1_bias_quan.npy")
print("conv1 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv1 parameter load
conv1 set done
	Total time: 0.10 sec


Pool1 memory map
---
Max Pool 1  
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0610_0000 ~ 0x061F_FFFF      
&nbsp;&nbsp;&nbsp;Size: 1024KB   

Conv2 memory map
---
Convolution 2  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0220_0000 ~ 0x026F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 5120KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0270_0000 ~ 0x027F_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0620_0000 ~ 0x062F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [9]:
print("conv2 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x0220_0000}, "./data/cifar10_network_quan_param/cifar10_conv2_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x0270_0000}, "./data/cifar10_network_quan_param/cifar10_conv2_bias_quan.npy")
print("conv2 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv2 parameter load
conv2 set done
	Total time: 1.95 sec


Pool2 memory map
---
Max Pool 2   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0630_0000 ~ 0x063F_FFFF      
&nbsp;&nbsp;&nbsp;Size: 1024KB 

Conv3 memory map
---
Convolution 3  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0280_0000 ~ 0x028F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 5120KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x02C0_0000 ~ 0x02CF_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0640_0000 ~ 0x064F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [10]:
print("conv3 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x0280_0000}, "./data/cifar10_network_quan_param/cifar10_conv3_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x02C0_0000}, "./data/cifar10_network_quan_param/cifar10_conv3_bias_quan.npy")
print("conv3 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv3 parameter load
conv3 set done
	Total time: 7.81 sec


Conv4 memory map
---
Convolution 4  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0300_0000 ~ 0x038F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 9216KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0390_0000 ~ 0x039F_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0650_0000 ~ 0x065F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [11]:
print("conv4 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x0300_0000}, "./data/cifar10_network_quan_param/cifar10_conv4_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x0390_0000}, "./data/cifar10_network_quan_param/cifar10_conv4_bias_quan.npy")
print("conv4 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv4 parameter load
conv4 set done
	Total time: 15.53 sec


Pool3 memory map
---
Max Pool 3   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0660_0000 ~ 0x066F_FFFF      
&nbsp;&nbsp;&nbsp;Size: 1024KB   

Conv5 memory map
---
Convolution 5  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x03A0_0000 ~ 0x03EF_FFFF  
&nbsp;&nbsp;&nbsp;Size: 5120KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x03F0_0000 ~ 0x03FF_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0670_0000 ~ 0x067F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [12]:
print("conv5 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x03A0_0000}, "./data/cifar10_network_quan_param/cifar10_conv5_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x03F0_0000}, "./data/cifar10_network_quan_param/cifar10_conv5_bias_quan.npy")
print("conv5 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv5 parameter load
conv5 set done
	Total time: 31.07 sec


Conv6 memory map
---
Convolution 6  
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0400_0000 ~ 0x048F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 9216KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0490_0000 ~ 0x049F_FFFF   
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0680_0000 ~ 0x068F_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB   

In [13]:
print("conv6 parameter load")
start = time.time()
SU.su_set_conv_w({'BASE_ADDR': 0x0400_0000}, "./data/cifar10_network_quan_param/cifar10_conv6_weight_quan.npy")
SU.su_set_conv_b({'BASE_ADDR': 0x0490_0000}, "./data/cifar10_network_quan_param/cifar10_conv6_bias_quan.npy")
print("conv6 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

conv6 parameter load
conv6 set done
	Total time: 61.98 sec


Pool4 memory map
---
Max Pool 4   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x0690_0000 ~ 0x069F_FFFF      
&nbsp;&nbsp;&nbsp;Size: 1024KB   

FC1 memory map
---
Fully-Connected 1    
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0500_0000 ~ 0x052F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 3072KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0530_0000 ~ 0x053F_FFFF     
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x06A0_0000 ~ 0x06AF_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB 

In [14]:
print("fc1 parameter load")
start = time.time()
SU.su_set_fc_w({'BASE_ADDR': 0x0500_0000}, "./data/cifar10_network_quan_param/cifar10_fc1_weight_quan.npy")
SU.su_set_fc_b({'BASE_ADDR': 0x0530_0000}, "./data/cifar10_network_quan_param/cifar10_fc1_bias_quan.npy")
print("fc1 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

fc1 parameter load
fc1 set done
	Total time: 27.53 sec


FC2 memory map
---
Fully-Connected 2    
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0540_0000 ~ 0x054F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 1024KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0550_0000 ~ 0x055F_FFFF     
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x06B0_0000 ~ 0x06BF_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB 

In [15]:
print("fc2 parameter load")
start = time.time()
SU.su_set_fc_w({'BASE_ADDR': 0x0540_0000}, "./data/cifar10_network_quan_param/cifar10_fc2_weight_quan.npy")
SU.su_set_fc_b({'BASE_ADDR': 0x0550_0000}, "./data/cifar10_network_quan_param/cifar10_fc2_bias_quan.npy")
print("fc2 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

fc2 parameter load
fc2 set done
	Total time: 1.74 sec


FC3 memory map
---
Fully-Connected 3    
Weight   
&nbsp;&nbsp;&nbsp;Address range: 0x0560_0000 ~ 0x056F_FFFF  
&nbsp;&nbsp;&nbsp;Size: 1024KB   
bias   
&nbsp;&nbsp;&nbsp;Address range: 0x0570_0000 ~ 0x057F_FFFF     
&nbsp;&nbsp;&nbsp;Size: 1024KB   
output   
&nbsp;&nbsp;&nbsp;Addresss range: 0x06C0_0000 ~ 0x06CF_FFFF       
&nbsp;&nbsp;&nbsp;Size: 1024KB 

In [16]:
print("fc3 parameter load")
start = time.time()
SU.su_set_fc_w({'BASE_ADDR': 0x0560_0000}, "./data/cifar10_network_quan_param/cifar10_fc3_weight_quan.npy")
SU.su_set_fc_b({'BASE_ADDR': 0x0570_0000}, "./data/cifar10_network_quan_param/cifar10_fc3_bias_quan.npy")
print("fc3 set done")
print("\tTotal time: {:.2f} sec".format(time.time() - start))

fc3 parameter load
fc3 set done
	Total time: 0.07 sec


### Parameter check (For debugging!)

In the below code, it verifies that **the data is stored correctly in DRAM**

In [17]:
debug_data = np.load("./data/cifar10_dataset_quan/images_100.npy")

In [18]:
print(debug_data.shape)

(100, 3, 32, 32)


### First, just check first image (debug_data[0])

In [19]:
# Print in 4 Bytes
debug_flat = debug_data.flatten()
for i in range(int(1 * 3 * 32 * 32 / 4)):
    temp = debug_flat[i*4:i*4+4]
    print(i, "\t", temp)

0 	 [0.578125 0.59375  0.703125 0.71875 ]
1 	 [0.609375 0.546875 0.65625  0.59375 ]
2 	 [0.578125 0.59375  0.625    0.609375]
3 	 [0.625    0.71875  0.765625 0.78125 ]
4 	 [0.734375 0.65625  0.609375 0.609375]
5 	 [0.546875 0.421875 0.4375   0.40625 ]
6 	 [0.421875 0.328125 0.265625 0.28125 ]
7 	 [ 0.328125  0.21875   0.03125  -0.125   ]
8 	 [0.484375 0.453125 0.59375  0.71875 ]
9 	 [0.65625  0.609375 0.6875   0.65625 ]
10 	 [0.671875 0.546875 0.53125  0.59375 ]
11 	 [0.671875 0.78125  0.796875 0.796875]
12 	 [0.765625 0.609375 0.515625 0.453125]
13 	 [0.359375 0.25     0.265625 0.28125 ]
14 	 [0.421875 0.390625 0.359375 0.3125  ]
15 	 [ 0.328125  0.203125  0.015625 -0.078125]
16 	 [0.453125 0.453125 0.578125 0.734375]
17 	 [0.609375 0.671875 0.703125 0.703125]
18 	 [0.671875 0.65625  0.578125 0.5625  ]
19 	 [0.625    0.71875  0.734375 0.765625]
20 	 [ 0.78125   0.59375   0.359375 -0.03125 ]
21 	 [-0.21875 -0.4375  -0.375   -0.15625]
22 	 [-0.0625    0.171875  0.328125  0.265625]
23 	 

In [20]:
debug_flat_bin = to_8bit_fixed_binary(debug_flat)
for i in range(int(1 * 3 * 32 * 32 / 4)):
    temp = debug_flat_bin[i*4:i*4+4]
    print(i, "\t", temp)

0 	 [37. 38. 45. 46.]
1 	 [39. 35. 42. 38.]
2 	 [37. 38. 40. 39.]
3 	 [40. 46. 49. 50.]
4 	 [47. 42. 39. 39.]
5 	 [35. 27. 28. 26.]
6 	 [27. 21. 17. 18.]
7 	 [ 21.  14.   2. 248.]
8 	 [31. 29. 38. 46.]
9 	 [42. 39. 44. 42.]
10 	 [43. 35. 34. 38.]
11 	 [43. 50. 51. 51.]
12 	 [49. 39. 33. 29.]
13 	 [23. 16. 17. 18.]
14 	 [27. 25. 23. 20.]
15 	 [ 21.  13.   1. 251.]
16 	 [29. 29. 37. 47.]
17 	 [39. 43. 45. 45.]
18 	 [43. 42. 37. 36.]
19 	 [40. 46. 47. 49.]
20 	 [ 50.  38.  23. 254.]
21 	 [242. 228. 232. 246.]
22 	 [252.  11.  21.  17.]
23 	 [ 20.  16.   6. 252.]
24 	 [34. 34. 39. 55.]
25 	 [47. 47. 49. 49.]
26 	 [45. 45. 47. 73.]
27 	 [58. 36. 42. 44.]
28 	 [ 37.  27. 235. 234.]
29 	 [228. 222. 209. 202.]
30 	 [215. 212. 245.   9.]
31 	 [17. 17. 13.  3.]
32 	 [34. 35. 40. 50.]
33 	 [49. 43. 49. 46.]
34 	 [ 44.  44.  54. 127.]
35 	 [78. 29. 24. 20.]
36 	 [243. 206. 214. 245.]
37 	 [244. 237. 227. 223.]
38 	 [202. 213. 214. 236.]
39 	 [ 4. 15. 10.  5.]
40 	 [26. 10.  6. 25.]
41 	 [40. 45. 4

In [21]:
# Check for written data in DRAM
base_addr_debug = 0x0000_0000 # input image
for i in range(int(1 * 3 * 32 * 32 / 4)):
    data = SU.su_read_data(base_addr_debug + i*4)
    print(data)

[46, 45, 38, 37]
[38, 42, 35, 39]
[39, 40, 38, 37]
[50, 49, 46, 40]
[39, 39, 42, 47]
[26, 28, 27, 35]
[18, 17, 21, 27]
[248, 2, 14, 21]
[46, 38, 29, 31]
[42, 44, 39, 42]
[38, 34, 35, 43]
[51, 51, 50, 43]
[29, 33, 39, 49]
[18, 17, 16, 23]
[20, 23, 25, 27]
[251, 1, 13, 21]
[47, 37, 29, 29]
[45, 45, 43, 39]
[36, 37, 42, 43]
[49, 47, 46, 40]
[254, 23, 38, 50]
[246, 232, 228, 242]
[17, 21, 11, 252]
[252, 6, 16, 20]
[55, 39, 34, 34]
[49, 49, 47, 47]
[73, 47, 45, 45]
[44, 42, 36, 58]
[234, 235, 27, 37]
[202, 209, 222, 228]
[9, 245, 212, 215]
[3, 13, 17, 17]
[50, 40, 35, 34]
[46, 49, 43, 49]
[127, 54, 44, 44]
[20, 24, 29, 78]
[245, 214, 206, 243]
[223, 227, 237, 244]
[236, 214, 213, 202]
[5, 10, 15, 4]
[25, 6, 10, 26]
[47, 47, 45, 40]
[61, 43, 45, 43]
[193, 227, 4, 36]
[250, 218, 193, 197]
[224, 246, 251, 255]
[194, 185, 221, 229]
[11, 15, 17, 239]
[217, 172, 240, 3]
[50, 48, 50, 32]
[25, 44, 46, 49]
[195, 231, 3, 5]
[9, 212, 200, 206]
[238, 236, 0, 24]
[171, 190, 214, 247]
[11, 18, 9, 208]
[1

[227, 229, 234, 243]
[62, 74, 90, 244]
[106, 252, 33, 68]
[29, 120, 127, 127]
[18, 18, 27, 17]
[218, 224, 230, 8]
[230, 221, 220, 220]
[243, 234, 228, 232]
[233, 235, 235, 247]
[64, 57, 23, 226]
[127, 55, 60, 73]
[225, 22, 127, 127]
[207, 210, 211, 216]
[218, 217, 217, 215]
[224, 226, 227, 223]
[234, 232, 230, 226]
[227, 233, 232, 234]
[41, 241, 224, 228]
[127, 81, 22, 70]
[220, 232, 47, 127]
[220, 219, 221, 220]
[215, 218, 220, 223]
[226, 221, 223, 219]
[239, 242, 242, 229]
[242, 232, 237, 241]
[245, 221, 232, 251]
[127, 108, 22, 35]
[225, 228, 230, 91]
[218, 219, 227, 227]
[219, 216, 217, 218]
[233, 229, 226, 223]
[247, 248, 249, 247]
[23, 7, 244, 247]
[219, 223, 237, 254]
[127, 126, 32, 233]
[232, 224, 229, 29]
[219, 218, 218, 229]
[225, 223, 219, 220]
[249, 245, 235, 227]
[228, 233, 235, 244]
[9, 24, 15, 247]
[223, 226, 236, 249]
[125, 115, 245, 219]
[224, 232, 243, 10]
[218, 216, 224, 235]
[226, 226, 224, 220]
[241, 230, 229, 235]
[239, 221, 230, 239]
[234, 236, 10, 14]
[225, 229,

### INFERENCE

### Just one image step by step

In [22]:
###################################################################
#        Convolution 1 + ReLU
###################################################################
# Convolution
# - in:       (n, 3, 32, 32)
# - out:     (n, 32, 28, 28)
# - weight:    (32, 3, 3, 3)
# - bias:               (32)
# ReLU
# - in:      (n. 32. 32. 32)
# - out:     (n. 32. 32. 32)
###################################################################
I = {'IN_CH': 3, 'OUT_CH': 32, 'FLEN': 32}
F = {'BASE_ADDR': 0x0000_0000, 'STRIDE_SIZE': 3*32*32, 'HSIZE': 3*32*32, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0200_0000, 'STRIDE_SIZE': 3*32*9, 'HSIZE': 3*32*9, 'VSIZE': 1}
B = {'BASE_ADDR': 0x0210_0000, 'STRIDE_SIZE': 32, 'HSIZE': 32, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0600_0000, 'STRIDE_SIZE': 32*32*32, 'HSIZE': 32*32*32, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [23]:
# You can check the result of first layer by below code
a = 0x0600_0000
for i in range(int(32*32*32/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 8 == 0):
        print(' -- ', int(i/8))

[0, 8, 9, 0][2, 18, 4, 8][7, 2, 8, 8][13, 9, 2, 0][0, 5, 12, 4][0, 6, 4, 13][9, 14, 12, 11][10, 7, 12, 72] --  0
[0, 2, 0, 0][1, 4, 0, 0][0, 4, 0, 0][0, 0, 0, 3][6, 1, 0, 0][0, 0, 0, 3][0, 12, 0, 0][0, 4, 6, 11] --  1
[8, 7, 0, 0][17, 1, 0, 0][0, 6, 19, 20][0, 0, 7, 4][0, 0, 0, 9][10, 13, 0, 0][0, 0, 13, 20][5, 2, 0, 8] --  2
[19, 10, 0, 0][0, 0, 0, 0][0, 9, 87, 14][0, 0, 0, 0][0, 0, 82, 62][23, 0, 0, 4][0, 32, 5, 16][9, 0, 0, 6] --  3
[13, 0, 0, 0][0, 0, 0, 0][1, 0, 20, 0][0, 0, 0, 0][0, 104, 105, 0][0, 0, 0, 0][22, 0, 0, 7][33, 11, 2, 10] --  4
[0, 0, 0, 39][44, 0, 0, 2][4, 0, 0, 0][89, 41, 0, 72][85, 46, 0, 0][0, 0, 0, 47][0, 0, 2, 0][25, 14, 1, 0] --  5
[15, 0, 55, 88][27, 0, 0, 1][0, 0, 0, 0][93, 35, 0, 27][0, 0, 0, 0][0, 0, 11, 15][0, 14, 0, 2][5, 10, 17, 8] --  6
[14, 0, 97, 55][0, 0, 8, 7][7, 0, 0, 82][48, 0, 0, 0][0, 0, 0, 0][0, 8, 0, 0][8, 30, 35, 0][0, 13, 34, 7] --  7
[0, 0, 127, 0][0, 0, 0, 37][120, 1, 35, 0][0, 0, 0, 5][0, 0, 0, 41][37, 18, 0, 0][19, 26, 18, 0][0, 0, 10, 

[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  66
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  67
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 41][16, 13, 6, 7][18, 0, 0, 0][0, 0, 0, 0] --  68
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 6, 55, 54][0, 0, 0, 0][7, 0, 0, 0][0, 0, 0, 0] --  69
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 3][51, 24, 38, 51][0, 0, 0, 0][0, 0, 0, 7][12, 0, 0, 0] --  70
[0, 0, 0, 75][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][30, 10, 2, 15][0, 0, 0, 0][0, 0, 0, 10][11, 18, 0, 0] --  71
[15, 0, 8, 99][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 4][10, 0, 0, 0][0, 8, 15, 20][1, 16, 2, 0] --  72
[14, 0, 26, 53][0, 0, 0, 0][39, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 56][65, 0, 0, 0][0, 25, 9, 18][6, 11, 18, 0] --  73
[4, 0, 17, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 6, 125][79, 0, 0, 0][0, 25, 4, 3][13, 19, 32, 11] --  74
[10, 0, 26, 0][0, 0, 

[96, 0, 0, 50][13, 0, 0, 0][57, 33, 0, 0][0, 0, 0, 0][0, 17, 0, 0][80, 24, 43, 0][0, 0, 0, 0][0, 0, 8, 0] --  138
[113, 0, 0, 18][11, 0, 0, 18][0, 0, 0, 0][0, 0, 0, 0][0, 17, 0, 0][127, 14, 11, 0][0, 0, 0, 0][0, 0, 16, 0] --  139
[112, 0, 0, 88][9, 0, 0, 0][0, 53, 0, 0][0, 17, 0, 0][0, 17, 0, 66][77, 0, 0, 0][0, 0, 0, 0][0, 0, 3, 0] --  140
[93, 0, 15, 105][0, 0, 0, 0][0, 38, 0, 0][69, 78, 0, 0][10, 7, 0, 61][52, 0, 0, 0][0, 0, 0, 0][1, 0, 0, 0] --  141
[75, 0, 45, 33][0, 0, 0, 0][0, 0, 0, 0][84, 21, 0, 0][10, 0, 0, 58][14, 0, 2, 12][0, 0, 0, 14][26, 0, 0, 0] --  142
[64, 0, 30, 0][0, 0, 0, 0][0, 0, 0, 127][99, 0, 0, 0][0, 0, 0, 49][0, 0, 0, 0][0, 0, 4, 24][0, 0, 0, 0] --  143
[62, 0, 8, 0][20, 0, 0, 0][15, 0, 64, 87][0, 0, 0, 0][0, 0, 0, 127][0, 0, 12, 23][0, 3, 0, 0][0, 0, 4, 0] --  144
[72, 0, 5, 1][1, 0, 0, 0][0, 0, 94, 0][0, 0, 0, 4][0, 0, 20, 0][0, 16, 17, 0][0, 0, 0, 0][0, 13, 9, 0] --  145
[68, 0, 0, 0][0, 0, 0, 28][0, 0, 29, 0][0, 0, 0, 42][10, 0, 5, 0][31, 93, 0, 0][0, 0, 0, 

[0, 18, 23, 0][0, 20, 127, 0][0, 3, 0, 127][0, 49, 37, 0][74, 117, 0, 8][1, 0, 10, 0][72, 0, 60, 0][0, 0, 0, 22] --  210
[0, 0, 19, 0][0, 0, 127, 0][0, 22, 0, 127][0, 79, 0, 0][16, 126, 0, 0][0, 0, 36, 12][55, 0, 53, 0][0, 9, 0, 32] --  211
[5, 0, 20, 9][0, 0, 127, 0][0, 0, 10, 79][0, 34, 0, 0][0, 69, 0, 0][0, 0, 0, 19][0, 0, 0, 0][0, 13, 0, 51] --  212
[127, 0, 15, 18][0, 0, 127, 0][0, 0, 40, 88][49, 0, 0, 0][0, 52, 0, 20][4, 7, 0, 6][0, 18, 0, 0][0, 5, 0, 62] --  213
[127, 0, 0, 9][0, 0, 127, 0][0, 0, 0, 112][23, 0, 0, 0][0, 13, 0, 16][5, 6, 0, 0][0, 0, 0, 0][0, 7, 0, 74] --  214
[113, 0, 4, 0][0, 88, 127, 0][0, 0, 29, 51][0, 0, 0, 0][0, 0, 0, 3][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 56] --  215
[12, 13, 47, 0][0, 127, 0, 0][0, 20, 49, 21][0, 0, 0, 2][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 8, 0, 0] --  216
[0, 22, 56, 13][0, 127, 0, 0][0, 126, 11, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 14][0, 4, 0, 0] --  217
[0, 9, 27, 17][28, 127, 0, 0][39, 91, 0, 0][0, 1, 4, 0][0, 0, 0, 0][0, 

[18, 0, 8, 3][0, 78, 84, 35][0, 0, 78, 21][19, 0, 0, 0][0, 0, 5, 17][0, 0, 12, 32][36, 0, 0, 0][0, 27, 85, 0] --  281
[0, 0, 0, 0][4, 88, 28, 0][0, 76, 19, 0][0, 0, 8, 5][0, 5, 14, 0][21, 20, 7, 1][0, 0, 0, 0][78, 98, 0, 0] --  282
[0, 0, 8, 1][29, 73, 0, 0][0, 30, 0, 0][10, 0, 12, 11][9, 0, 0, 13][0, 0, 5, 8][0, 0, 0, 112][43, 0, 0, 29] --  283
[7, 7, 0, 12][1, 59, 0, 0][0, 28, 0, 12][0, 2, 2, 0][0, 13, 23, 10][0, 18, 4, 0][0, 8, 80, 16][0, 0, 16, 58] --  284
[0, 0, 0, 16][0, 2, 0, 0][113, 0, 1, 40][11, 9, 0, 0][12, 1, 0, 0][24, 0, 0, 0][22, 39, 46, 0][12, 0, 57, 0] --  285
[0, 11, 7, 1][2, 2, 8, 0][26, 0, 2, 0][0, 17, 23, 19][0, 0, 12, 28][0, 0, 10, 8][35, 37, 13, 0][0, 0, 55, 0] --  286
[0, 0, 5, 0][10, 0, 0, 6][0, 5, 20, 1][12, 0, 0, 7][39, 49, 44, 0][0, 8, 1, 30][51, 24, 0, 0][0, 34, 0, 96] --  287
[56, 66, 62, 58][61, 66, 67, 68][70, 74, 68, 58][59, 67, 72, 75][79, 78, 75, 69][63, 59, 58, 58][58, 56, 54, 55][58, 62, 62, 49] --  288
[69, 83, 76, 65][67, 75, 79, 80][83, 86, 74, 60]

...

[0, 0, 0, 5][0, 0, 2, 0][0, 9, 73, 31][0, 0, 0, 0][0, 35, 29, 36][0, 0, 0, 0][0, 0, 0, 28][17, 0, 3, 0] --  707
[0, 0, 0, 2][0, 0, 2, 0][0, 0, 0, 127][22, 0, 0, 16][0, 0, 46, 41][4, 11, 0, 0][0, 0, 0, 0][43, 11, 0, 0] --  708
[0, 0, 0, 83][46, 0, 0, 1][0, 0, 0, 0][91, 8, 0, 0][20, 0, 0, 26][0, 0, 19, 20][0, 0, 0, 0][31, 55, 0, 0] --  709
[0, 0, 0, 4][57, 0, 0, 11][0, 0, 0, 0][44, 41, 0, 0][0, 0, 0, 71][22, 0, 0, 16][20, 0, 0, 0][0, 58, 18, 0] --  710
[0, 0, 0, 0][0, 0, 0, 0][16, 0, 0, 21][13, 4, 0, 0][13, 0, 0, 13][127, 13, 0, 0][26, 33, 0, 0][0, 31, 51, 0] --  711
[105, 0, 0, 0][0, 0, 0, 45][47, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][75, 78, 0, 0][0, 39, 9, 0][0, 0, 93, 16] --  712
[85, 81, 0, 0][0, 0, 0, 42][127, 110, 0, 0][0, 0, 26, 0][0, 0, 0, 12][3, 0, 5, 0][0, 0, 0, 13][0, 0, 1, 72] --  713
[56, 0, 68, 0][0, 0, 0, 0][111, 127, 73, 0][55, 0, 0, 30][0, 0, 0, 46][23, 0, 8, 0][0, 43, 0, 0][0, 0, 0, 12] --  714
[94, 0, 32, 32][8, 0, 29, 0][0, 127, 62, 32][99, 60, 0, 0][20, 0, 0, 14][47, 0, 

[0, 127, 0, 0][83, 73, 0, 0][0, 0, 127, 12][0, 97, 108, 64][34, 47, 18, 0][0, 24, 89, 100][11, 0, 38, 72][67, 0, 0, 0] --  777
[0, 117, 0, 0][121, 94, 29, 0][0, 0, 127, 52][0, 99, 103, 66][52, 85, 0, 0][0, 58, 106, 127][2, 0, 28, 65][85, 5, 0, 0] --  778
[9, 108, 0, 0][127, 90, 43, 0][0, 0, 127, 19][0, 75, 127, 70][33, 117, 0, 0][0, 119, 127, 124][0, 0, 15, 45][58, 31, 0, 0] --  779
[29, 114, 0, 0][127, 55, 19, 86][0, 0, 67, 0][0, 38, 127, 79][19, 127, 0, 0][0, 127, 127, 91][7, 0, 0, 0][32, 29, 27, 0] --  780
[28, 86, 0, 70][127, 11, 0, 103][0, 0, 0, 0][0, 48, 127, 69][36, 127, 0, 0][17, 127, 110, 74][26, 0, 0, 0][20, 40, 34, 0] --  781
[0, 55, 0, 117][127, 6, 0, 91][0, 0, 0, 0][12, 127, 89, 48][65, 114, 0, 0][16, 121, 101, 84][22, 0, 0, 0][33, 56, 0, 0] --  782
[0, 53, 0, 73][127, 49, 0, 64][31, 0, 0, 0][122, 127, 52, 28][54, 70, 0, 0][2, 79, 99, 98][22, 0, 0, 0][42, 19, 0, 0] --  783
[0, 69, 0, 10][127, 107, 0, 0][88, 0, 0, 0][108, 117, 40, 12][25, 0, 0, 0][0, 0, 95, 127][31, 0, 0, 0

[0, 11, 58, 0][0, 0, 0, 0][0, 0, 0, 27][11, 0, 0, 0][7, 0, 18, 33][0, 0, 0, 0][4, 5, 16, 25][30, 41, 50, 45] --  845
[0, 19, 45, 0][0, 0, 0, 0][0, 6, 32, 45][0, 0, 0, 0][0, 0, 33, 38][0, 0, 0, 0][0, 10, 23, 28][28, 39, 60, 48] --  846
[1, 7, 21, 0][0, 0, 0, 0][0, 11, 22, 0][0, 0, 0, 0][0, 2, 65, 67][19, 0, 0, 0][0, 4, 5, 11][25, 60, 84, 38] --  847
[7, 0, 6, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 5, 50, 6][0, 0, 0, 0][0, 0, 0, 2][34, 62, 46, 0] --  848
[12, 0, 1, 0][0, 0, 9, 0][0, 0, 0, 0][0, 0, 12, 5][0, 9, 3, 0][0, 0, 0, 0][0, 1, 43, 77][86, 65, 25, 0] --  849
[10, 0, 7, 0][0, 0, 1, 0][0, 0, 0, 0][5, 2, 34, 28][6, 14, 0, 0][4, 0, 0, 0][0, 52, 103, 123][97, 55, 28, 0] --  850
[0, 0, 17, 17][1, 0, 2, 9][0, 0, 0, 0][0, 0, 46, 29][8, 19, 0, 0][0, 0, 0, 0][0, 19, 32, 20][0, 0, 0, 0] --  851
[0, 0, 2, 9][0, 0, 0, 66][80, 48, 24, 23][38, 54, 42, 0][0, 10, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  852
[0, 0, 0, 0][0, 0, 30, 117][115, 75, 55, 35][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 

[87, 55, 0, 0][0, 0, 20, 23][0, 0, 0, 0][27, 46, 49, 39][4, 2, 0, 0][0, 0, 0, 0][0, 0, 9, 12][0, 0, 0, 33] --  918
[57, 102, 69, 18][0, 0, 23, 0][0, 0, 0, 19][47, 47, 48, 44][20, 6, 0, 1][3, 0, 0, 5][9, 1, 0, 5][0, 0, 0, 29] --  919
[0, 75, 118, 93][0, 28, 54, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 4][10, 6, 5, 7][5, 0, 0, 0][0, 0, 3, 4] --  920
[0, 0, 61, 97][45, 84, 36, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][2, 2, 0, 0][0, 0, 0, 4][7, 3, 0, 0] --  921
[0, 0, 14, 55][72, 127, 49, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 2, 0][0, 5, 22, 41][23, 0, 0, 0] --  922
[0, 0, 0, 10][48, 127, 116, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 6, 21, 14][2, 5, 22, 22][4, 0, 32, 50] --  923
[0, 0, 0, 0][10, 101, 127, 0][0, 0, 3, 9][0, 0, 0, 0][0, 0, 0, 0][0, 0, 2, 12][18, 11, 7, 0][0, 0, 39, 63] --  924
[0, 0, 0, 0][0, 24, 88, 31][1, 7, 29, 30][18, 9, 0, 0][0, 0, 0, 0][0, 0, 0, 12][23, 15, 0, 0][0, 0, 4, 64] --  925
[0, 0, 0, 0][0, 0, 3, 3][0, 0, 13, 16][24, 28, 20, 22][18, 11, 2, 0][0, 0, 0, 9][25,

[0, 8, 3, 4][17, 24, 18, 0][0, 0, 0, 0][0, 0, 1, 1][0, 0, 0, 0][0, 0, 3, 2][0, 0, 0, 0][3, 26, 17, 127] --  989
[0, 16, 4, 0][8, 16, 13, 0][0, 0, 0, 0][0, 0, 2, 4][0, 0, 0, 0][0, 2, 1, 0][0, 0, 0, 0][5, 17, 13, 127] --  990
[0, 70, 56, 49][53, 60, 60, 45][38, 38, 39, 41][40, 40, 50, 48][35, 29, 30, 40][50, 48, 46, 42][37, 41, 45, 50][59, 59, 64, 127] --  991
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 6, 11, 0] --  992
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][12, 18, 11, 3][0, 0, 0, 0][0, 0, 0, 0] --  993
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 10, 5][2, 12, 27, 29][32, 18, 0, 0][0, 0, 0, 0] --  994
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][37, 50, 6, 0][0, 0, 0, 1][14, 16, 9, 0][0, 0, 0, 0] --  995
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][3, 21, 45, 65][71, 33, 0, 0][0, 0, 0, 0][0, 8, 35, 14][0, 0, 0, 0] --  996
[0, 43, 54, 0][0, 0, 0, 0][0, 0, 0, 8][17, 11, 35, 29][12, 0, 0, 0][0, 0, 0, 0][0, 12, 41, 31

In [ ]:
#print([temp[3],temp[2],temp[1],temp[0]])

In [24]:
###################################################################
#        Max Pool 1
###################################################################
# Max Pooling
# - in:      (n. 32. 32. 32)
# - out:     (n, 32, 16, 16)
###################################################################
I = {'IN_CH': 32, 'FLEN': 32}
F = {'BASE_ADDR': 0x0600_0000, 'STRIDE_SIZE': 32*32*32, 'HSIZE': 32*32*32, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0610_0000, 'STRIDE_SIZE': 32*16*16, 'HSIZE': 32*16*16, 'VSIZE': 1}
SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)

1

In [25]:
a = 0x0610_0000
for i in range(int(32*16*16/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 4 == 0):
        print('--',int(i/4))

[8, 9, 18, 8][7, 8, 13, 3][6, 12, 6, 13][14, 12, 10, 72]-- 0
[19, 0, 17, 0][9, 87, 0, 7][0, 82, 23, 4][32, 20, 9, 8]-- 1
[13, 39, 44, 2][4, 20, 89, 72][104, 105, 0, 47][22, 7, 33, 10]-- 2
[15, 97, 27, 8][7, 82, 93, 27][0, 0, 8, 15][30, 35, 13, 34]-- 3
[10, 127, 0, 75][120, 35, 0, 23][0, 113, 37, 0][26, 18, 0, 27]-- 4
[74, 9, 30, 3][5, 49, 56, 2][26, 127, 0, 53][33, 11, 19, 23]-- 5
[119, 36, 59, 21][10, 127, 22, 25][0, 103, 3, 44][25, 55, 15, 38]-- 6
[97, 5, 34, 19][32, 111, 19, 78][42, 74, 45, 19][24, 0, 87, 58]-- 7
[16, 0, 16, 52][32, 15, 115, 93][54, 36, 127, 56][42, 56, 70, 0]-- 8
[27, 0, 0, 22][30, 65, 126, 70][0, 66, 0, 71][87, 35, 0, 0]-- 9
[37, 0, 0, 127][7, 89, 70, 0][15, 38, 122, 77][0, 0, 0, 33]-- 10
[0, 43, 65, 127][0, 121, 36, 53][85, 72, 0, 0][14, 0, 8, 31]-- 11
[0, 20, 127, 22][127, 127, 8, 2][5, 15, 6, 12][16, 0, 50, 49]-- 12
[5, 5, 50, 0][114, 14, 4, 2][5, 0, 0, 0][14, 61, 35, 102]-- 13
[0, 1, 0, 20][127, 18, 11, 20][16, 5, 0, 31][63, 23, 0, 76]-- 14
[26, 28, 23, 30][52

[61, 0, 8, 0][17, 111, 13, 14][70, 93, 39, 24][72, 3, 20, 14]-- 130
[88, 80, 73, 0][51, 1, 79, 12][26, 46, 17, 57][0, 49, 7, 29]-- 131
[70, 111, 0, 31][127, 72, 81, 54][25, 19, 67, 58][21, 11, 38, 25]-- 132
[32, 117, 0, 123][102, 5, 44, 0][25, 127, 106, 3][32, 23, 11, 22]-- 133
[0, 106, 26, 13][34, 127, 56, 0][23, 79, 0, 26][14, 43, 16, 50]-- 134
[63, 32, 77, 34][124, 127, 65, 61][24, 93, 27, 37][31, 35, 9, 66]-- 135
[42, 0, 50, 55][93, 127, 70, 75][3, 127, 127, 81][14, 28, 82, 21]-- 136
[36, 0, 51, 14][119, 0, 119, 127][27, 64, 107, 42][80, 48, 34, 2]-- 137
[78, 2, 17, 127][52, 104, 127, 68][10, 0, 52, 53][78, 81, 10, 23]-- 138
[55, 10, 45, 117][116, 24, 0, 3][52, 63, 22, 0][16, 20, 24, 29]-- 139
[18, 55, 78, 127][14, 78, 19, 7][0, 17, 6, 32][36, 4, 27, 89]-- 140
[0, 8, 88, 28][76, 19, 10, 12][9, 14, 21, 8][0, 112, 98, 29]-- 141
[7, 16, 59, 0][113, 40, 11, 2][13, 23, 24, 4][39, 80, 12, 58]-- 142
[11, 7, 10, 8][26, 20, 17, 23][49, 44, 8, 30][51, 13, 34, 96]-- 143
[83, 76, 75, 80][86, 7

[34, 82, 8, 0][4, 127, 127, 77][7, 0, 10, 23][2, 12, 4, 1]-- 258
[0, 0, 0, 0][0, 0, 0, 35][26, 50, 38, 14][13, 0, 7, 0]-- 259
[27, 19, 9, 35][67, 92, 8, 0][5, 13, 0, 0][0, 0, 7, 19]-- 260
[39, 39, 22, 73][127, 62, 0, 0][0, 0, 47, 29][20, 11, 0, 0]-- 261
[16, 0, 50, 0][33, 0, 69, 86][0, 54, 0, 0][4, 0, 0, 4]-- 262
[22, 0, 0, 0][31, 127, 127, 49][43, 17, 4, 70][50, 34, 17, 0]-- 263
[14, 0, 38, 0][0, 38, 20, 0][0, 63, 115, 73][67, 12, 0, 4]-- 264
[27, 11, 0, 11][36, 73, 18, 0][19, 125, 127, 6][24, 37, 5, 0]-- 265
[0, 0, 26, 50][50, 55, 127, 127][127, 55, 0, 13][5, 3, 2, 7]-- 266
[67, 82, 67, 76][89, 127, 108, 60][9, 9, 26, 8][5, 0, 0, 4]-- 267
[0, 29, 28, 19][49, 0, 0, 3][0, 0, 0, 0][27, 32, 0, 28]-- 268
[0, 0, 56, 74][50, 0, 0, 0][4, 0, 10, 30][0, 0, 58, 78]-- 269
[5, 0, 0, 41][66, 37, 46, 41][12, 0, 0, 0][0, 0, 0, 5]-- 270
[0, 0, 0, 0][0, 0, 0, 37][44, 19, 14, 0][33, 43, 20, 0]-- 271
[48, 0, 11, 0][11, 0, 5, 3][1, 0, 0, 2][16, 18, 1, 0]-- 272
[46, 1, 7, 5][1, 105, 80, 16][17, 22, 55, 82

[30, 0, 0, 3][2, 0, 100, 10][56, 37, 23, 32][9, 0, 8, 15]-- 385
[41, 31, 0, 5][0, 0, 127, 81][61, 0, 36, 46][77, 59, 0, 17]-- 386
[127, 66, 2, 0][23, 53, 29, 105][12, 0, 93, 68][95, 79, 0, 0]-- 387
[127, 0, 83, 0][14, 127, 97, 120][47, 18, 54, 100][24, 72, 67, 0]-- 388
[117, 0, 127, 43][0, 127, 99, 127][117, 0, 119, 127][2, 65, 85, 0]-- 389
[114, 70, 127, 103][0, 67, 48, 127][127, 0, 127, 127][26, 0, 40, 34]-- 390
[55, 117, 127, 91][31, 0, 127, 89][114, 0, 121, 101][22, 0, 56, 0]-- 391
[76, 10, 127, 0][101, 8, 117, 40][44, 75, 0, 127][31, 0, 8, 1]-- 392
[59, 15, 127, 26][87, 127, 4, 43][127, 63, 126, 111][0, 0, 23, 13]-- 393
[0, 47, 127, 28][127, 127, 0, 54][127, 97, 72, 16][0, 0, 34, 19]-- 394
[0, 5, 127, 0][127, 127, 15, 43][93, 64, 0, 0][0, 0, 20, 8]-- 395
[53, 0, 3, 39][127, 127, 14, 9][6, 5, 0, 0][0, 8, 2, 0]-- 396
[58, 19, 0, 127][127, 6, 26, 10][0, 0, 0, 0][23, 8, 51, 42]-- 397
[67, 15, 6, 115][127, 4, 8, 17][0, 0, 25, 18][0, 35, 111, 22]-- 398
[64, 19, 11, 12][38, 0, 0, 4][0, 0

In [26]:
###################################################################
#        Convolution 2 + ReLU
###################################################################
# Convolution
# - in:       (n, 32, 16, 16)
# - out:      (n, 64, 16, 16)
# - weight:    (64, 32, 3, 3)
# - bias:                (64)
# ReLU
# - in:       (n. 64. 16. 16)
# - out:      (n. 64. 16. 16)
###################################################################
I = {'IN_CH': 32, 'OUT_CH': 64, 'FLEN': 16}
F = {'BASE_ADDR': 0x0610_0000, 'STRIDE_SIZE': 32*16*16, 'HSIZE': 32*16*16, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0220_0000, 'STRIDE_SIZE': 32*64*9, 'HSIZE': 32*64*9, 'VSIZE': 1}
B = {'BASE_ADDR': 0x0270_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0620_0000, 'STRIDE_SIZE': 64*16*16, 'HSIZE': 64*16*16, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [27]:
a = 0x0620_0000
for i in range(int(64*16*16/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 4 == 0):
        print(' -- ', int(i/4))

[0, 7, 18, 18][19, 0, 6, 25][23, 28, 47, 57][39, 20, 0, 0] --  0
[4, 23, 32, 41][43, 0, 10, 45][30, 28, 43, 69][63, 53, 0, 0] --  1
[0, 22, 37, 53][0, 0, 0, 4][0, 0, 0, 0][0, 0, 0, 0] --  2
[0, 0, 0, 40][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  3
[0, 0, 0, 6][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  4
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  5
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  6
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  7
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  8
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 6, 0] --  9
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][4, 45, 76, 36] --  10
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 2, 35][45, 49, 36, 0] --  11
[0, 0, 0, 0][0, 0, 0, 6][0, 10, 45, 67][70, 64, 54, 0] --  12
[0, 0, 0, 0][0, 0, 21, 70][76, 81, 75, 73][58, 40, 19, 0] --  13
[0, 0, 0, 0][0, 0, 85, 94][87, 78, 67, 62][40, 3, 0, 0] --  14
[0, 12, 0, 0][0, 0, 31, 37][43, 46, 35, 22][16, 0, 0, 0] --  15
[0, 37, 29, 22][19, 44, 19, 11][26, 67, 47, 1][2, 1

[0, 2, 79, 0][0, 0, 0, 0][0, 0, 69, 89][0, 0, 0, 0] --  133
[0, 41, 61, 0][0, 0, 0, 0][0, 0, 127, 27][0, 0, 0, 0] --  134
[0, 59, 8, 0][0, 0, 63, 0][0, 1, 49, 0][0, 3, 39, 19] --  135
[0, 0, 0, 0][0, 21, 4, 0][0, 1, 0, 26][0, 0, 15, 56] --  136
[0, 0, 0, 0][0, 28, 0, 10][0, 0, 4, 0][0, 0, 31, 62] --  137
[0, 0, 0, 0][0, 0, 0, 54][0, 0, 0, 0][0, 39, 56, 38] --  138
[0, 0, 0, 0][88, 22, 1, 7][0, 0, 0, 0][38, 48, 54, 5] --  139
[1, 0, 0, 7][127, 22, 0, 0][0, 0, 7, 9][23, 21, 17, 57] --  140
[0, 0, 0, 68][115, 0, 0, 3][10, 16, 16, 17][25, 3, 90, 95] --  141
[20, 0, 6, 38][0, 0, 0, 0][0, 0, 33, 27][0, 68, 127, 0] --  142
[10, 0, 39, 3][0, 0, 0, 0][0, 0, 23, 0][3, 98, 33, 0] --  143
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 9, 16][0, 0, 0, 0] --  144
[23, 0, 0, 0][0, 0, 58, 0][0, 0, 0, 16][0, 5, 0, 0] --  145
[19, 0, 0, 0][0, 0, 52, 0][0, 0, 18, 0][0, 0, 0, 0] --  146
[0, 0, 18, 0][0, 0, 52, 0][0, 0, 31, 0][0, 0, 0, 0] --  147
[0, 0, 42, 0][0, 0, 0, 32][0, 0, 17, 36][0, 0, 0, 0] --  148
[0, 0, 113, 0][

[16, 13, 0, 57][7, 0, 0, 0][0, 0, 0, 0][0, 12, 4, 0] --  263
[0, 0, 0, 27][0, 0, 0, 30][0, 9, 0, 0][0, 5, 15, 2] --  264
[0, 0, 0, 0][58, 21, 78, 82][0, 25, 0, 0][0, 25, 27, 0] --  265
[0, 0, 1, 0][19, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  266
[0, 0, 10, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  267
[0, 0, 0, 0][0, 0, 0, 11][0, 0, 0, 0][0, 0, 0, 0] --  268
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  269
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  270
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  271
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  272
[0, 0, 0, 2][0, 0, 12, 0][0, 0, 29, 0][0, 0, 0, 0] --  273
[0, 0, 9, 8][0, 0, 0, 0][0, 43, 66, 0][0, 0, 0, 0] --  274
[0, 0, 74, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 11] --  275
[0, 0, 6, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 8] --  276
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 38, 0][0, 0, 0, 0] --  277
[0, 0, 0, 0][0, 0, 9, 0][0, 0, 47, 0][0, 0, 0, 0] --  278
[0, 0, 0, 0][0, 0, 58, 0][0, 0, 0, 0][0, 0, 0, 38] --  279
[0, 

[0, 0, 0, 0][0, 0, 2, 13][17, 28, 22, 11][39, 54, 4, 0] --  397
[0, 0, 0, 0][0, 0, 0, 8][51, 74, 39, 2][43, 50, 0, 0] --  398
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  399
[56, 32, 29, 23][92, 88, 82, 49][72, 58, 33, 43][38, 34, 58, 39] --  400
[73, 29, 13, 4][101, 109, 103, 68][81, 35, 0, 14][35, 48, 26, 2] --  401
[80, 75, 32, 30][66, 83, 32, 24][48, 39, 38, 39][13, 66, 44, 13] --  402
[49, 82, 117, 107][56, 74, 2, 11][2, 52, 23, 55][0, 0, 31, 0] --  403
[43, 59, 127, 127][99, 127, 60, 29][0, 42, 0, 40][0, 0, 1, 0] --  404
[65, 42, 78, 127][85, 113, 1, 15][0, 22, 3, 28][0, 0, 0, 0] --  405
[58, 34, 58, 39][85, 88, 50, 10][0, 0, 0, 0][0, 0, 0, 0] --  406
[53, 3, 82, 45][81, 55, 88, 53][0, 78, 17, 26][20, 16, 34, 0] --  407
[3, 0, 74, 52][64, 89, 73, 60][46, 52, 48, 29][33, 31, 37, 0] --  408
[0, 8, 16, 24][10, 87, 67, 27][60, 25, 0, 0][0, 17, 35, 8] --  409
[45, 50, 62, 85][21, 38, 0, 0][0, 0, 0, 0][0, 0, 0, 12] --  410
[35, 95, 78, 104][51, 21, 0, 0][0, 0, 0, 0][0, 0, 0, 1

[6, 32, 82, 127][76, 51, 20, 14][0, 0, 0, 9][0, 0, 15, 3] --  527
[23, 23, 24, 23][57, 69, 97, 26][34, 12, 0, 0][0, 7, 41, 53] --  528
[14, 35, 25, 34][79, 72, 80, 16][8, 5, 11, 0][0, 0, 27, 72] --  529
[1, 22, 31, 85][105, 76, 51, 41][0, 28, 58, 37][18, 0, 50, 75] --  530
[8, 11, 92, 113][113, 127, 82, 72][25, 24, 56, 61][33, 0, 34, 59] --  531
[8, 24, 127, 112][88, 102, 98, 108][39, 40, 114, 106][49, 25, 20, 41] --  532
[28, 37, 111, 80][70, 88, 127, 95][48, 28, 71, 54][56, 23, 42, 54] --  533
[36, 44, 44, 95][127, 127, 127, 68][56, 27, 61, 43][43, 55, 73, 74] --  534
[42, 58, 12, 121][127, 72, 39, 26][84, 61, 86, 58][35, 42, 63, 87] --  535
[50, 90, 3, 119][94, 70, 40, 83][86, 25, 45, 32][5, 35, 53, 79] --  536
[52, 96, 56, 83][97, 60, 56, 84][11, 0, 0, 0][0, 0, 15, 39] --  537
[14, 53, 82, 97][70, 19, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  538
[0, 0, 20, 69][35, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  539
[0, 0, 0, 23][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  540
[0, 0, 0, 0][0, 0, 0, 0][0,

[0, 30, 109, 32][0, 90, 19, 3][6, 5, 39, 93][32, 0, 106, 122] --  660
[0, 0, 127, 127][0, 40, 103, 84][32, 0, 0, 5][37, 9, 69, 127] --  661
[0, 0, 43, 127][21, 0, 9, 103][88, 0, 0, 1][1, 5, 12, 59] --  662
[0, 0, 10, 127][44, 0, 26, 22][47, 0, 0, 20][29, 0, 0, 0] --  663
[0, 34, 24, 127][127, 0, 0, 0][0, 4, 0, 0][24, 0, 0, 0] --  664
[0, 18, 27, 93][127, 127, 0, 0][56, 40, 28, 0][0, 36, 13, 34] --  665
[0, 56, 0, 0][0, 72, 0, 0][69, 127, 0, 0][0, 5, 39, 119] --  666
[0, 65, 51, 0][0, 0, 0, 0][0, 33, 25, 0][0, 0, 0, 88] --  667
[0, 45, 13, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 28] --  668
[0, 35, 0, 8][0, 0, 0, 7][0, 0, 0, 0][0, 0, 0, 0] --  669
[0, 0, 23, 13][25, 0, 0, 12][0, 0, 0, 0][0, 0, 0, 0] --  670
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  671
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 2, 0, 0] --  672
[0, 3, 0, 0][7, 12, 20, 2][0, 0, 0, 0][0, 23, 41, 7] --  673
[0, 0, 0, 0][7, 9, 18, 2][11, 31, 39, 16][8, 1, 40, 25] --  674
[0, 0, 5, 18][32, 37, 28, 42][33, 29, 69, 80][58,

[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  794
[0, 0, 0, 0][0, 0, 0, 0][0, 10, 0, 0][0, 0, 0, 0] --  795
[27, 0, 0, 0][20, 40, 49, 39][12, 0, 0, 0][0, 0, 0, 0] --  796
[0, 25, 0, 0][7, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  797
[0, 0, 0, 12][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  798
[0, 0, 0, 90][8, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  799
[0, 9, 0, 2][22, 0, 0, 0][0, 0, 0, 0][11, 4, 10, 38] --  800
[0, 6, 9, 17][22, 17, 0, 0][0, 2, 0, 0][0, 31, 45, 34] --  801
[0, 0, 4, 15][0, 18, 0, 0][0, 50, 0, 0][0, 0, 84, 25] --  802
[0, 0, 0, 57][0, 0, 32, 0][0, 45, 31, 0][21, 0, 72, 71] --  803
[0, 0, 0, 2][44, 0, 0, 0][0, 0, 39, 0][41, 23, 0, 90] --  804
[0, 0, 0, 0][71, 0, 10, 0][0, 0, 0, 0][18, 83, 0, 11] --  805
[0, 0, 0, 0][0, 0, 0, 20][0, 0, 0, 0][0, 38, 10, 0] --  806
[0, 0, 0, 3][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  807
[4, 61, 16, 46][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  808
[44, 10, 42, 36][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  809
[52, 6, 0, 0][0, 0, 0, 0][0, 0, 0, 0

[1, 0, 0, 0][28, 8, 1, 0][11, 0, 0, 0][0, 0, 28, 0] --  928
[4, 0, 0, 0][23, 0, 0, 0][14, 0, 0, 0][0, 0, 0, 0] --  929
[72, 37, 0, 1][11, 0, 0, 0][0, 0, 0, 0][0, 1, 0, 0] --  930
[40, 0, 0, 17][0, 0, 0, 0][0, 0, 0, 0][0, 12, 7, 0] --  931
[15, 0, 0, 35][0, 0, 0, 0][72, 1, 0, 0][0, 0, 33, 0] --  932
[16, 0, 0, 0][0, 0, 0, 0][61, 0, 0, 0][0, 0, 24, 0] --  933
[2, 0, 0, 0][0, 0, 0, 0][38, 0, 0, 0][0, 0, 29, 0] --  934
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 4][0, 0, 0, 0] --  935
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  936
[0, 0, 46, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  937
[0, 0, 63, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  938
[12, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  939
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  940
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  941
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 7][0, 0, 0, 0] --  942
[0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0][0, 0, 0, 0] --  943
[0, 0, 5, 14][5, 0, 0, 19][23, 18, 29, 36][24, 6, 0, 0] --  944
[

In [28]:
###################################################################
#        Max Pool 2
###################################################################
# Max Pooling
# - in:      (n. 64. 16. 16)
# - out:       (n, 64, 8, 8)
###################################################################
I = {'IN_CH': 64, 'FLEN': 16}
F = {'BASE_ADDR': 0x0620_0000, 'STRIDE_SIZE': 64*16*16, 'HSIZE': 64*16*16, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0630_0000, 'STRIDE_SIZE': 64*8*8, 'HSIZE': 64*8*8, 'VSIZE': 1}
SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)

1

In [29]:
a = 0x0630_0000
for i in range(int(64*8*8/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 2 == 0):
        print('--',int(i/2))

[110, 127, 127, 127][127, 113, 127, 127]-- 0
[0, 79, 72, 26][32, 0, 52, 51]-- 1
[0, 70, 95, 53][11, 0, 0, 0]-- 2
[23, 44, 79, 77][69, 21, 0, 0]-- 3
[30, 43, 85, 87][25, 3, 6, 0]-- 4
[7, 35, 39, 16][0, 31, 0, 0]-- 5
[11, 35, 10, 0][0, 0, 9, 0]-- 6
[0, 39, 35, 38][0, 30, 40, 0]-- 7
[78, 0, 0, 0][0, 0, 0, 0]-- 8
[63, 0, 0, 0][0, 16, 0, 0]-- 9
[98, 23, 13, 19][60, 51, 40, 127]-- 10
[127, 50, 40, 42][27, 0, 58, 127]-- 11
[118, 1, 2, 0][15, 0, 0, 12]-- 12
[111, 25, 1, 0][8, 17, 43, 71]-- 13
[81, 9, 3, 21][43, 58, 64, 91]-- 14
[82, 63, 63, 64][93, 87, 92, 51]-- 15
[30, 21, 0, 0][0, 0, 0, 0]-- 16
[5, 0, 0, 0][0, 96, 71, 0]-- 17
[2, 50, 54, 23][0, 64, 115, 0]-- 18
[3, 54, 62, 0][12, 48, 0, 0]-- 19
[66, 88, 32, 0][0, 11, 0, 0]-- 20
[79, 4, 71, 0][44, 0, 0, 0]-- 21
[24, 0, 4, 0][0, 0, 0, 3]-- 22
[0, 0, 1, 14][0, 0, 0, 0]-- 23
[0, 0, 0, 0][0, 0, 0, 0]-- 24
[0, 24, 14, 13][111, 127, 127, 28]-- 25
[114, 127, 127, 111][47, 0, 3, 93]-- 26
[0, 0, 0, 0][0, 0, 0, 40]-- 27
[0, 0, 0, 0][0, 0, 30, 52]-- 28


[0, 0, 0, 0][0, 0, 0, 0]-- 241
[19, 23, 34, 11][0, 0, 0, 42]-- 242
[0, 0, 0, 0][0, 0, 0, 40]-- 243
[0, 0, 0, 0][0, 0, 1, 10]-- 244
[0, 0, 0, 0][0, 32, 53, 61]-- 245
[11, 0, 0, 0][9, 15, 0, 0]-- 246
[38, 0, 0, 29][21, 9, 4, 46]-- 247
[53, 62, 50, 44][39, 54, 37, 28]-- 248
[0, 62, 44, 56][92, 127, 127, 34]-- 249
[86, 127, 127, 127][117, 127, 127, 127]-- 250
[81, 102, 111, 92][91, 69, 60, 119]-- 251
[61, 96, 65, 77][74, 106, 62, 99]-- 252
[65, 75, 87, 68][104, 96, 88, 50]-- 253
[58, 59, 90, 90][96, 79, 61, 6]-- 254
[43, 48, 72, 66][24, 38, 39, 0]-- 255
[0, 0, 0, 0][0, 0, 0, 0]-- 256
[0, 0, 0, 0][0, 23, 0, 0]-- 257
[6, 4, 3, 25][45, 72, 23, 0]-- 258
[46, 59, 47, 62][40, 0, 75, 13]-- 259
[28, 14, 0, 0][0, 3, 36, 8]-- 260
[0, 0, 18, 0][69, 0, 4, 58]-- 261
[11, 5, 45, 17][0, 12, 22, 50]-- 262
[53, 66, 57, 41][43, 40, 49, 63]-- 263
[43, 0, 0, 0][10, 5, 5, 0]-- 264
[61, 57, 46, 52][18, 0, 50, 34]-- 265
[0, 0, 0, 0][0, 12, 78, 14]-- 266
[14, 0, 0, 0][0, 0, 58, 31]-- 267
[4, 4, 0, 0][0, 37, 25, 0

[57, 58, 48, 71][112, 116, 127, 89]-- 482
[111, 120, 102, 84][73, 91, 127, 127]-- 483
[127, 115, 66, 44][35, 90, 88, 127]-- 484
[126, 115, 71, 78][97, 51, 58, 84]-- 485
[64, 58, 84, 91][73, 66, 69, 19]-- 486
[26, 11, 39, 43][64, 51, 43, 50]-- 487
[0, 27, 31, 40][37, 51, 56, 62]-- 488
[0, 69, 55, 13][13, 0, 6, 48]-- 489
[1, 55, 53, 14][0, 0, 0, 17]-- 490
[0, 0, 0, 0][0, 22, 0, 0]-- 491
[0, 0, 0, 22][6, 0, 0, 26]-- 492
[0, 0, 3, 0][0, 7, 0, 0]-- 493
[16, 3, 0, 0][0, 0, 0, 0]-- 494
[0, 0, 0, 0][0, 0, 0, 0]-- 495
[0, 0, 0, 0][0, 0, 0, 0]-- 496
[4, 27, 20, 0][0, 0, 0, 14]-- 497
[0, 0, 0, 0][0, 0, 0, 0]-- 498
[0, 0, 0, 0][0, 18, 0, 0]-- 499
[0, 0, 0, 0][0, 0, 0, 0]-- 500
[0, 0, 0, 0][0, 0, 0, 0]-- 501
[0, 0, 0, 0][0, 0, 0, 0]-- 502
[0, 0, 6, 17][5, 1, 0, 0]-- 503
[118, 72, 51, 52][38, 0, 6, 26]-- 504
[102, 64, 54, 58][1, 0, 2, 30]-- 505
[21, 0, 0, 0][36, 86, 73, 19]-- 506
[15, 0, 0, 0][31, 84, 54, 0]-- 507
[0, 0, 8, 18][61, 33, 43, 22]-- 508
[44, 23, 11, 13][6, 3, 27, 30]-- 509
[0, 0, 0, 13]

In [30]:
###################################################################
#        Convolution 3 + ReLU
###################################################################
# Convolution
# - in:        (n, 64, 8, 8)
# - out:      (n, 128, 8, 8)
# - weight:  (128, 64, 3, 3)
# - bias:              (128)
# ReLU
# - in:       (n. 128. 8. 8)
# - out:      (n. 128. 8. 8)
###################################################################
I = {'IN_CH': 64, 'OUT_CH': 128, 'FLEN': 8}
F = {'BASE_ADDR': 0x0630_0000, 'STRIDE_SIZE': 64*8*8, 'HSIZE': 64*8*8, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0280_0000, 'STRIDE_SIZE': int(64*128*9/2), 'HSIZE': int(64*128*9/2), 'VSIZE': 2}
B = {'BASE_ADDR': 0x02C0_0000, 'STRIDE_SIZE': 128, 'HSIZE': 128, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0640_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [31]:
a = 0x0640_0000
for i in range(int(128*8*8/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 2 == 0):
        print(' -- ', int(i/2))

[0, 0, 0, 0][0, 0, 0, 0] --  0
[31, 16, 0, 0][0, 0, 0, 0] --  1
[13, 28, 6, 0][0, 20, 11, 0] --  2
[1, 37, 6, 0][0, 40, 44, 54] --  3
[0, 1, 0, 0][12, 0, 6, 76] --  4
[0, 0, 23, 0][0, 0, 0, 50] --  5
[2, 29, 30, 0][30, 32, 16, 67] --  6
[0, 7, 0, 0][42, 22, 0, 25] --  7
[0, 0, 0, 0][0, 0, 0, 0] --  8
[0, 0, 0, 0][0, 20, 0, 0] --  9
[0, 0, 0, 0][0, 0, 0, 13] --  10
[0, 0, 0, 0][0, 0, 0, 19] --  11
[0, 0, 0, 0][0, 0, 0, 2] --  12
[0, 0, 0, 0][0, 0, 24, 44] --  13
[0, 54, 3, 0][0, 21, 61, 67] --  14
[0, 0, 0, 0][0, 0, 0, 43] --  15
[0, 0, 0, 0][0, 0, 0, 0] --  16
[8, 0, 0, 0][0, 0, 0, 0] --  17
[0, 0, 0, 0][0, 0, 0, 0] --  18
[0, 0, 0, 0][0, 0, 0, 0] --  19
[32, 13, 0, 0][0, 0, 4, 0] --  20
[1, 0, 0, 0][0, 0, 16, 0] --  21
[7, 0, 0, 0][23, 2, 0, 0] --  22
[8, 0, 2, 0][0, 3, 11, 0] --  23
[0, 3, 0, 0][0, 0, 0, 0] --  24
[0, 0, 0, 0][0, 0, 0, 0] --  25
[0, 0, 0, 0][0, 0, 0, 0] --  26
[2, 35, 0, 0][6, 0, 0, 62] --  27
[24, 11, 0, 0][0, 0, 0, 20] --  28
[0, 0, 0, 0][0, 0, 8, 24] --  29
[55, 4

[0, 0, 30, 38][0, 0, 0, 0] --  241
[0, 0, 30, 24][9, 16, 0, 0] --  242
[40, 44, 86, 69][43, 90, 23, 0] --  243
[0, 0, 29, 39][5, 25, 9, 0] --  244
[0, 0, 34, 6][0, 0, 0, 0] --  245
[0, 0, 24, 0][0, 0, 0, 0] --  246
[0, 0, 35, 36][0, 0, 0, 0] --  247
[0, 15, 28, 0][0, 0, 0, 14] --  248
[0, 0, 0, 0][0, 0, 0, 0] --  249
[0, 0, 0, 10][0, 0, 0, 0] --  250
[0, 0, 0, 0][0, 14, 3, 0] --  251
[0, 10, 0, 0][0, 90, 87, 0] --  252
[0, 2, 0, 0][0, 3, 0, 0] --  253
[0, 2, 2, 0][0, 0, 0, 0] --  254
[0, 0, 0, 0][0, 0, 0, 0] --  255
[0, 0, 0, 0][0, 0, 13, 24] --  256
[0, 0, 0, 0][0, 0, 0, 0] --  257
[0, 0, 0, 0][0, 0, 0, 0] --  258
[13, 0, 0, 0][0, 0, 87, 1] --  259
[0, 0, 0, 0][0, 0, 0, 0] --  260
[0, 0, 0, 0][0, 0, 0, 0] --  261
[0, 0, 0, 0][0, 0, 0, 0] --  262
[18, 0, 0, 0][17, 18, 24, 16] --  263
[45, 4, 18, 29][1, 0, 0, 19] --  264
[8, 0, 8, 23][0, 0, 0, 0] --  265
[19, 0, 0, 43][26, 0, 0, 0] --  266
[62, 0, 38, 51][31, 29, 31, 0] --  267
[78, 18, 53, 36][1, 50, 99, 0] --  268
[78, 16, 22, 33][24,

[1, 29, 97, 75][79, 92, 68, 3] --  476
[43, 49, 77, 52][54, 60, 4, 0] --  477
[17, 14, 25, 16][0, 0, 0, 0] --  478
[11, 0, 9, 0][0, 0, 0, 0] --  479
[0, 0, 4, 0][0, 0, 0, 3] --  480
[0, 6, 0, 0][0, 0, 0, 0] --  481
[0, 0, 0, 0][0, 0, 0, 0] --  482
[0, 0, 0, 0][0, 0, 0, 0] --  483
[0, 0, 0, 0][0, 0, 0, 0] --  484
[0, 0, 0, 0][0, 0, 0, 0] --  485
[0, 0, 0, 0][0, 0, 0, 0] --  486
[26, 23, 19, 9][0, 13, 0, 0] --  487
[17, 52, 2, 2][0, 0, 0, 0] --  488
[0, 60, 0, 0][0, 0, 0, 0] --  489
[0, 16, 0, 0][0, 0, 0, 0] --  490
[0, 0, 0, 0][0, 0, 0, 0] --  491
[9, 0, 0, 0][0, 0, 0, 0] --  492
[28, 27, 0, 0][0, 0, 0, 0] --  493
[16, 38, 14, 0][0, 0, 0, 0] --  494
[5, 23, 0, 0][0, 15, 0, 0] --  495
[0, 0, 0, 0][0, 0, 0, 0] --  496
[0, 9, 17, 30][53, 54, 1, 0] --  497
[37, 71, 41, 26][49, 49, 6, 0] --  498
[83, 95, 20, 0][23, 74, 5, 0] --  499
[61, 86, 42, 19][29, 105, 43, 7] --  500
[13, 70, 56, 23][56, 62, 76, 39] --  501
[17, 54, 46, 34][40, 83, 63, 10] --  502
[0, 0, 0, 0][0, 15, 0, 0] --  503
[0, 

[38, 69, 65, 29][20, 48, 42, 2] --  711
[4, 0, 0, 0][0, 0, 0, 0] --  712
[0, 0, 5, 3][2, 0, 0, 0] --  713
[54, 37, 23, 28][56, 70, 26, 9] --  714
[23, 0, 0, 0][3, 51, 57, 0] --  715
[13, 0, 0, 0][5, 0, 5, 55] --  716
[0, 0, 0, 3][0, 2, 59, 89] --  717
[0, 26, 19, 46][112, 87, 46, 33] --  718
[23, 29, 33, 48][66, 33, 14, 44] --  719
[1, 0, 0, 0][0, 12, 0, 0] --  720
[20, 0, 0, 0][1, 18, 0, 0] --  721
[37, 48, 37, 20][26, 0, 47, 14] --  722
[26, 42, 11, 32][63, 20, 27, 80] --  723
[24, 30, 4, 0][34, 15, 0, 16] --  724
[9, 0, 0, 14][46, 25, 9, 0] --  725
[0, 0, 26, 40][44, 3, 15, 35] --  726
[14, 1, 12, 10][0, 0, 0, 0] --  727
[34, 92, 0, 18][1, 30, 14, 0] --  728
[14, 71, 0, 8][0, 0, 6, 0] --  729
[26, 39, 28, 0][0, 0, 19, 37] --  730
[38, 14, 0, 0][0, 0, 40, 5] --  731
[5, 29, 2, 0][0, 25, 0, 0] --  732
[36, 10, 0, 0][0, 0, 0, 0] --  733
[102, 13, 0, 0][0, 8, 0, 0] --  734
[26, 21, 0, 0][0, 24, 0, 0] --  735
[0, 0, 0, 0][0, 0, 0, 0] --  736
[71, 63, 16, 12][0, 0, 39, 33] --  737
[5, 0, 

[60, 93, 76, 46][45, 5, 0, 0] --  940
[60, 31, 25, 0][16, 26, 0, 15] --  941
[33, 0, 0, 14][36, 15, 0, 32] --  942
[6, 20, 18, 14][0, 0, 0, 15] --  943
[75, 127, 76, 44][29, 54, 59, 22] --  944
[15, 74, 16, 10][21, 36, 8, 0] --  945
[0, 27, 25, 4][0, 0, 3, 76] --  946
[16, 37, 33, 12][5, 28, 0, 94] --  947
[70, 69, 38, 8][43, 12, 0, 11] --  948
[86, 17, 26, 3][19, 0, 0, 0] --  949
[11, 0, 0, 0][0, 0, 0, 0] --  950
[0, 0, 0, 0][0, 0, 0, 8] --  951
[0, 0, 0, 0][0, 0, 0, 36] --  952
[0, 0, 0, 60][68, 0, 0, 0] --  953
[29, 22, 37, 70][72, 27, 0, 0] --  954
[57, 23, 15, 35][41, 6, 18, 7] --  955
[11, 0, 0, 26][4, 0, 74, 45] --  956
[5, 0, 34, 34][35, 57, 70, 44] --  957
[30, 16, 46, 36][1, 0, 0, 16] --  958
[0, 0, 4, 4][0, 0, 0, 0] --  959
[0, 0, 0, 17][29, 11, 0, 7] --  960
[0, 0, 33, 0][0, 0, 0, 15] --  961
[0, 0, 0, 0][0, 0, 0, 0] --  962
[0, 0, 0, 0][0, 4, 0, 0] --  963
[0, 0, 0, 0][6, 0, 0, 0] --  964
[0, 0, 0, 0][0, 0, 0, 0] --  965
[0, 0, 0, 0][0, 0, 0, 0] --  966
[0, 0, 0, 0][0, 0, 

In [32]:
###################################################################
#        Convolution 4 + ReLU
###################################################################
# Convolution
# - in:       (n, 128, 8, 8)
# - out:      (n, 128, 8, 8)
# - weight: (128, 128, 3, 3)
# - bias:              (128)
# ReLU
# - in:       (n. 128. 8. 8)
# - out:      (n. 128. 8. 8)
###################################################################
I = {'IN_CH': 128, 'OUT_CH': 128, 'FLEN': 8}
F = {'BASE_ADDR': 0x0640_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0300_0000, 'STRIDE_SIZE': int(128*128*9/4), 'HSIZE': int(128*128*9/4), 'VSIZE': 4}
B = {'BASE_ADDR': 0x0390_0000, 'STRIDE_SIZE': 128, 'HSIZE': 128, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0650_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [33]:
a = 0x0650_0000
for i in range(int(128*8*8/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end='')
    if ((i+1) % 2 == 0):
        print('--',int(i/2))

[6, 0, 0, 0][0, 0, 0, 0]-- 0
[0, 0, 0, 0][0, 0, 0, 0]-- 1
[25, 1, 0, 0][17, 2, 0, 0]-- 2
[26, 0, 0, 0][0, 0, 0, 0]-- 3
[0, 0, 0, 0][0, 0, 0, 0]-- 4
[0, 0, 0, 0][0, 0, 0, 0]-- 5
[0, 0, 17, 14][0, 0, 0, 0]-- 6
[0, 0, 31, 17][0, 0, 7, 0]-- 7
[0, 0, 0, 0][0, 5, 0, 0]-- 8
[63, 91, 47, 11][0, 75, 101, 0]-- 9
[0, 67, 46, 0][0, 32, 72, 25]-- 10
[0, 18, 9, 0][0, 0, 0, 16]-- 11
[0, 0, 0, 0][0, 0, 0, 0]-- 12
[0, 8, 0, 0][0, 0, 0, 0]-- 13
[0, 0, 0, 0][0, 0, 0, 0]-- 14
[0, 0, 0, 0][0, 0, 0, 0]-- 15
[0, 0, 0, 0][0, 0, 36, 0]-- 16
[0, 2, 13, 0][0, 0, 72, 78]-- 17
[0, 0, 0, 0][0, 0, 0, 92]-- 18
[0, 0, 0, 0][0, 0, 0, 3]-- 19
[0, 0, 0, 0][0, 0, 0, 0]-- 20
[6, 0, 0, 0][0, 16, 0, 0]-- 21
[41, 22, 0, 0][0, 59, 47, 0]-- 22
[23, 51, 34, 35][33, 68, 64, 12]-- 23
[0, 0, 0, 0][0, 0, 0, 0]-- 24
[0, 0, 0, 0][0, 0, 0, 0]-- 25
[0, 0, 0, 0][0, 0, 0, 0]-- 26
[0, 0, 0, 0][0, 0, 0, 0]-- 27
[0, 0, 0, 0][0, 0, 16, 6]-- 28
[0, 3, 7, 0][0, 0, 0, 0]-- 29
[6, 24, 21, 14][0, 0, 0, 2]-- 30
[0, 0, 0, 0][0, 0, 0, 0]-- 31
[0, 0, 

[10, 0, 0, 0][0, 0, 0, 0]-- 265
[0, 0, 0, 0][0, 0, 2, 0]-- 266
[0, 0, 0, 0][0, 0, 0, 0]-- 267
[0, 0, 0, 0][0, 0, 0, 0]-- 268
[0, 0, 0, 0][0, 0, 0, 0]-- 269
[0, 0, 0, 0][35, 0, 0, 0]-- 270
[0, 0, 0, 5][40, 0, 0, 0]-- 271
[0, 0, 0, 0][0, 0, 0, 0]-- 272
[0, 0, 0, 0][0, 0, 0, 0]-- 273
[0, 0, 0, 0][0, 20, 22, 0]-- 274
[0, 0, 0, 0][2, 0, 0, 23]-- 275
[0, 0, 0, 0][0, 0, 0, 28]-- 276
[0, 0, 0, 0][0, 0, 0, 22]-- 277
[0, 0, 0, 30][56, 12, 20, 14]-- 278
[0, 0, 1, 24][24, 10, 7, 0]-- 279
[0, 0, 2, 0][0, 0, 0, 4]-- 280
[0, 0, 0, 0][0, 0, 0, 0]-- 281
[0, 0, 0, 0][23, 17, 0, 0]-- 282
[0, 7, 30, 65][80, 51, 7, 0]-- 283
[0, 10, 31, 29][0, 0, 0, 0]-- 284
[0, 0, 0, 0][0, 0, 27, 2]-- 285
[0, 3, 0, 0][8, 32, 61, 5]-- 286
[0, 0, 9, 30][29, 34, 20, 0]-- 287
[0, 0, 0, 5][0, 0, 0, 6]-- 288
[0, 0, 2, 13][0, 0, 0, 0]-- 289
[0, 0, 0, 1][1, 0, 0, 0]-- 290
[0, 0, 0, 47][47, 31, 0, 0]-- 291
[0, 0, 56, 64][36, 62, 57, 0]-- 292
[0, 2, 38, 24][12, 23, 15, 0]-- 293
[0, 0, 11, 20][0, 0, 0, 0]-- 294
[0, 0, 5, 16][0, 0, 0,

[0, 0, 4, 0][3, 0, 0, 0]-- 525
[0, 0, 0, 0][0, 0, 0, 0]-- 526
[0, 0, 0, 0][0, 0, 0, 0]-- 527
[9, 29, 3, 0][0, 44, 29, 0]-- 528
[0, 0, 0, 0][0, 0, 28, 0]-- 529
[0, 0, 11, 0][0, 0, 19, 27]-- 530
[0, 0, 17, 0][0, 0, 0, 0]-- 531
[4, 0, 1, 0][0, 5, 0, 0]-- 532
[11, 0, 0, 0][14, 35, 0, 0]-- 533
[0, 14, 17, 0][12, 8, 0, 0]-- 534
[0, 0, 0, 0][0, 0, 0, 0]-- 535
[0, 0, 0, 0][25, 24, 0, 0]-- 536
[52, 26, 4, 29][89, 80, 0, 0]-- 537
[41, 34, 2, 0][0, 2, 32, 0]-- 538
[0, 0, 0, 0][0, 0, 37, 0]-- 539
[0, 0, 0, 0][0, 0, 0, 0]-- 540
[0, 0, 0, 0][0, 0, 0, 0]-- 541
[0, 0, 0, 0][0, 0, 0, 0]-- 542
[0, 0, 0, 0][43, 10, 0, 13]-- 543
[0, 0, 0, 0][23, 45, 0, 0]-- 544
[11, 23, 0, 0][17, 21, 5, 0]-- 545
[29, 58, 0, 0][0, 0, 0, 0]-- 546
[0, 0, 0, 0][0, 0, 0, 3]-- 547
[0, 0, 0, 0][0, 0, 0, 46]-- 548
[0, 0, 0, 0][0, 0, 0, 2]-- 549
[0, 0, 0, 0][0, 0, 0, 0]-- 550
[0, 0, 0, 0][0, 0, 0, 0]-- 551
[0, 0, 0, 0][0, 0, 0, 0]-- 552
[0, 0, 0, 0][0, 0, 0, 0]-- 553
[0, 0, 0, 0][1, 0, 0, 0]-- 554
[0, 0, 0, 0][0, 0, 0, 0]-- 555
[0

[0, 0, 0, 0][0, 0, 0, 0]-- 780
[0, 0, 0, 0][0, 0, 0, 0]-- 781
[0, 0, 0, 0][0, 0, 0, 0]-- 782
[0, 0, 0, 0][0, 2, 0, 0]-- 783
[9, 21, 17, 15][0, 11, 21, 23]-- 784
[46, 49, 33, 33][7, 7, 16, 31]-- 785
[45, 48, 47, 27][0, 0, 0, 8]-- 786
[0, 1, 25, 0][0, 0, 0, 0]-- 787
[0, 0, 18, 15][31, 13, 0, 4]-- 788
[30, 41, 54, 41][34, 9, 0, 1]-- 789
[61, 71, 65, 27][15, 0, 0, 0]-- 790
[24, 12, 13, 6][2, 0, 0, 0]-- 791
[45, 39, 0, 26][25, 39, 0, 0]-- 792
[51, 49, 11, 38][0, 3, 2, 0]-- 793
[7, 0, 0, 0][0, 0, 23, 0]-- 794
[0, 0, 0, 0][0, 0, 0, 0]-- 795
[0, 0, 0, 0][0, 0, 0, 0]-- 796
[16, 0, 0, 0][0, 0, 0, 0]-- 797
[28, 0, 0, 0][18, 0, 0, 0]-- 798
[19, 0, 0, 0][10, 0, 0, 0]-- 799
[0, 0, 0, 0][0, 0, 0, 0]-- 800
[0, 0, 0, 0][0, 0, 0, 0]-- 801
[0, 0, 0, 0][1, 0, 0, 0]-- 802
[0, 0, 0, 0][0, 15, 0, 0]-- 803
[0, 0, 0, 0][0, 0, 0, 0]-- 804
[0, 0, 0, 0][0, 0, 0, 29]-- 805
[0, 0, 0, 0][0, 0, 4, 20]-- 806
[0, 0, 0, 0][0, 0, 0, 0]-- 807
[0, 0, 0, 0][0, 10, 12, 0]-- 808
[0, 0, 0, 0][0, 43, 37, 0]-- 809
[0, 0, 0, 0][0

In [34]:
###################################################################
#        Max Pool 3
###################################################################
# Max Pooling
# - in:      (n. 128. 8. 8)
# - out:     (n, 128, 4, 4)
###################################################################
I = {'IN_CH': 128, 'FLEN': 8}
F = {'BASE_ADDR': 0x0650_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0660_0000, 'STRIDE_SIZE': 128*4*4, 'HSIZE': 128*4*4, 'VSIZE': 1}
SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)

1

In [35]:
a = 0x0660_0000
for i in range(int(128*4*4/4)):
    temp = SU.su_read_data(a + 4*i)
    print(i, [temp[3],temp[2],temp[1],temp[0]])

0 [6, 0, 0, 0]
1 [26, 0, 17, 0]
2 [0, 0, 0, 0]
3 [0, 31, 0, 7]
4 [91, 47, 75, 101]
5 [67, 46, 32, 72]
6 [8, 0, 0, 0]
7 [0, 0, 0, 0]
8 [2, 13, 0, 78]
9 [0, 0, 0, 92]
10 [6, 0, 16, 0]
11 [51, 35, 68, 64]
12 [0, 0, 0, 0]
13 [0, 0, 0, 0]
14 [3, 7, 0, 16]
15 [24, 21, 0, 2]
16 [0, 0, 33, 0]
17 [0, 2, 7, 19]
18 [0, 0, 0, 8]
19 [0, 0, 0, 0]
20 [0, 0, 0, 0]
21 [0, 0, 0, 0]
22 [0, 0, 0, 0]
23 [0, 0, 16, 2]
24 [0, 0, 8, 10]
25 [0, 0, 42, 22]
26 [1, 0, 25, 27]
27 [0, 0, 0, 0]
28 [49, 0, 0, 0]
29 [12, 0, 0, 68]
30 [0, 0, 0, 0]
31 [0, 0, 0, 1]
32 [0, 0, 0, 0]
33 [0, 0, 0, 0]
34 [0, 0, 0, 0]
35 [0, 0, 0, 0]
36 [0, 0, 0, 38]
37 [0, 0, 0, 12]
38 [0, 0, 0, 0]
39 [0, 0, 0, 0]
40 [0, 0, 38, 0]
41 [0, 0, 20, 48]
42 [0, 0, 3, 29]
43 [0, 0, 0, 0]
44 [0, 0, 0, 0]
45 [0, 10, 0, 0]
46 [1, 25, 32, 3]
47 [6, 39, 0, 0]
48 [28, 0, 0, 0]
49 [38, 17, 0, 3]
50 [0, 0, 0, 0]
51 [4, 5, 0, 0]
52 [0, 0, 14, 0]
53 [0, 0, 0, 28]
54 [0, 0, 0, 0]
55 [0, 0, 0, 0]
56 [28, 4, 0, 0]
57 [27, 49, 0, 0]
58 [11, 26, 39, 50]
59 [52, 17

459 [0, 16, 21, 0]
460 [0, 0, 0, 0]
461 [28, 38, 0, 0]
462 [36, 0, 0, 0]
463 [19, 0, 0, 3]
464 [0, 0, 0, 0]
465 [30, 0, 0, 8]
466 [0, 9, 26, 27]
467 [16, 0, 44, 44]
468 [5, 0, 0, 1]
469 [19, 0, 0, 21]
470 [0, 0, 0, 0]
471 [8, 0, 0, 0]
472 [3, 0, 11, 10]
473 [0, 0, 0, 0]
474 [0, 0, 0, 30]
475 [0, 15, 0, 0]
476 [0, 0, 0, 0]
477 [0, 10, 18, 0]
478 [0, 0, 0, 3]
479 [0, 0, 0, 0]
480 [0, 0, 0, 0]
481 [24, 17, 24, 5]
482 [0, 0, 0, 0]
483 [4, 0, 28, 24]
484 [0, 0, 0, 0]
485 [60, 55, 13, 41]
486 [16, 0, 0, 2]
487 [14, 0, 0, 0]
488 [0, 0, 0, 0]
489 [0, 0, 0, 6]
490 [0, 0, 0, 28]
491 [0, 0, 0, 6]
492 [0, 0, 0, 0]
493 [1, 13, 0, 0]
494 [0, 0, 0, 0]
495 [48, 11, 7, 0]
496 [18, 11, 23, 0]
497 [34, 0, 16, 0]
498 [25, 0, 0, 2]
499 [21, 0, 0, 2]
500 [0, 0, 12, 0]
501 [27, 31, 83, 0]
502 [37, 0, 0, 0]
503 [38, 3, 0, 0]
504 [0, 0, 0, 0]
505 [0, 0, 41, 0]
506 [2, 21, 3, 0]
507 [0, 12, 44, 57]
508 [5, 0, 17, 0]
509 [36, 12, 0, 27]
510 [0, 1, 15, 0]
511 [0, 0, 0, 0]


In [36]:
###################################################################
#        Convolution 5+ ReLU
###################################################################
# Convolution
# - in:       (n, 128, 4, 4)
# - out:      (n, 256, 4, 4)
# - weight: (256, 128, 3, 3)
# - bias:              (256)
# ReLU
# - in:       (n. 256. 4. 4)
# - out:      (n. 256. 4. 4)
###################################################################
I = {'IN_CH': 128, 'OUT_CH': 256, 'FLEN': 4}
F = {'BASE_ADDR': 0x0660_0000, 'STRIDE_SIZE': 128*4*4, 'HSIZE': 128*4*4, 'VSIZE': 1}
W = {'BASE_ADDR': 0x03A0_0000, 'STRIDE_SIZE': int(128*256*9/8), 'HSIZE': int(128*256*9/8), 'VSIZE': 8}
B = {'BASE_ADDR': 0x03F0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0670_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [37]:
a = 0x0670_0000
for i in range(int(256*4*4/4)):
    temp = SU.su_read_data(a + 4*i)
    print(i, [temp[3],temp[2],temp[1],temp[0]])

0 [0, 0, 0, 0]
1 [7, 0, 0, 0]
2 [0, 0, 0, 0]
3 [1, 0, 0, 0]
4 [8, 11, 6, 0]
5 [0, 0, 0, 0]
6 [0, 0, 0, 4]
7 [0, 0, 0, 0]
8 [0, 0, 0, 0]
9 [16, 0, 0, 0]
10 [13, 10, 20, 0]
11 [6, 2, 1, 0]
12 [0, 0, 0, 0]
13 [0, 0, 0, 0]
14 [0, 0, 0, 0]
15 [0, 0, 0, 0]
16 [0, 0, 0, 3]
17 [0, 0, 0, 0]
18 [0, 0, 0, 0]
19 [0, 0, 0, 0]
20 [0, 0, 5, 0]
21 [12, 16, 0, 0]
22 [0, 6, 0, 0]
23 [9, 0, 0, 0]
24 [5, 7, 3, 0]
25 [3, 0, 0, 0]
26 [0, 0, 0, 0]
27 [0, 0, 0, 0]
28 [0, 4, 11, 0]
29 [9, 38, 38, 14]
30 [2, 12, 8, 0]
31 [8, 15, 16, 0]
32 [0, 0, 0, 0]
33 [0, 0, 2, 0]
34 [0, 0, 3, 1]
35 [0, 0, 0, 0]
36 [2, 0, 20, 0]
37 [7, 0, 0, 0]
38 [4, 0, 0, 0]
39 [0, 0, 3, 1]
40 [8, 1, 3, 5]
41 [14, 9, 0, 15]
42 [7, 0, 2, 8]
43 [5, 0, 0, 2]
44 [0, 2, 0, 0]
45 [9, 11, 1, 6]
46 [8, 22, 26, 19]
47 [1, 9, 11, 0]
48 [13, 0, 0, 0]
49 [0, 0, 0, 0]
50 [0, 0, 0, 0]
51 [0, 0, 0, 0]
52 [4, 5, 9, 4]
53 [4, 8, 7, 7]
54 [5, 7, 5, 1]
55 [2, 4, 3, 2]
56 [0, 0, 0, 0]
57 [14, 0, 0, 0]
58 [5, 1, 0, 5]
59 [0, 0, 0, 0]
60 [0, 9, 0, 0]
61 [0, 0, 

482 [0, 7, 0, 0]
483 [10, 20, 15, 0]
484 [0, 0, 3, 16]
485 [0, 0, 0, 0]
486 [0, 0, 0, 0]
487 [0, 0, 0, 8]
488 [0, 0, 0, 0]
489 [0, 2, 17, 0]
490 [1, 27, 29, 14]
491 [0, 8, 2, 0]
492 [0, 0, 0, 0]
493 [0, 0, 0, 0]
494 [0, 0, 0, 0]
495 [0, 0, 0, 0]
496 [4, 0, 0, 0]
497 [0, 0, 0, 1]
498 [6, 0, 0, 7]
499 [0, 0, 0, 0]
500 [0, 0, 0, 0]
501 [7, 11, 0, 0]
502 [0, 0, 0, 0]
503 [23, 25, 18, 0]
504 [0, 0, 6, 0]
505 [0, 0, 0, 0]
506 [0, 13, 2, 0]
507 [0, 0, 0, 0]
508 [0, 0, 0, 0]
509 [0, 0, 0, 8]
510 [0, 0, 0, 10]
511 [0, 0, 6, 5]
512 [0, 0, 5, 0]
513 [0, 0, 0, 0]
514 [0, 0, 0, 0]
515 [0, 0, 0, 0]
516 [0, 0, 0, 0]
517 [0, 0, 5, 0]
518 [0, 0, 0, 0]
519 [0, 0, 0, 0]
520 [0, 0, 15, 5]
521 [11, 6, 2, 7]
522 [1, 0, 0, 0]
523 [0, 0, 0, 0]
524 [0, 0, 0, 0]
525 [0, 0, 0, 0]
526 [10, 0, 0, 0]
527 [0, 0, 0, 0]
528 [0, 0, 0, 7]
529 [0, 0, 0, 10]
530 [6, 0, 0, 7]
531 [0, 0, 0, 0]
532 [0, 0, 0, 0]
533 [0, 0, 0, 0]
534 [0, 0, 0, 0]
535 [0, 0, 0, 0]
536 [0, 15, 20, 14]
537 [0, 0, 0, 0]
538 [1, 0, 0, 0]
539 [0, 0,

954 [17, 3, 13, 2]
955 [17, 16, 11, 0]
956 [0, 3, 0, 2]
957 [0, 17, 3, 0]
958 [16, 22, 0, 0]
959 [10, 10, 0, 5]
960 [10, 27, 13, 3]
961 [0, 15, 0, 0]
962 [0, 9, 0, 0]
963 [0, 0, 0, 11]
964 [0, 0, 0, 0]
965 [6, 2, 0, 0]
966 [8, 3, 0, 0]
967 [3, 0, 0, 0]
968 [0, 0, 0, 0]
969 [13, 10, 0, 0]
970 [6, 0, 0, 0]
971 [8, 11, 11, 24]
972 [0, 0, 0, 0]
973 [0, 0, 14, 8]
974 [0, 0, 0, 6]
975 [0, 0, 0, 0]
976 [3, 0, 0, 0]
977 [0, 0, 0, 0]
978 [0, 0, 0, 0]
979 [0, 0, 0, 0]
980 [0, 0, 0, 0]
981 [6, 0, 0, 0]
982 [1, 0, 0, 0]
983 [0, 0, 4, 0]
984 [0, 0, 0, 0]
985 [0, 11, 2, 4]
986 [0, 11, 20, 15]
987 [13, 50, 38, 28]
988 [0, 0, 0, 2]
989 [0, 0, 0, 0]
990 [0, 0, 4, 0]
991 [12, 3, 22, 2]
992 [11, 9, 5, 0]
993 [29, 7, 0, 10]
994 [9, 1, 0, 0]
995 [8, 0, 6, 0]
996 [7, 16, 12, 0]
997 [8, 6, 2, 6]
998 [0, 0, 0, 16]
999 [0, 0, 0, 1]
1000 [5, 16, 5, 0]
1001 [5, 0, 25, 0]
1002 [6, 7, 30, 4]
1003 [0, 0, 4, 2]
1004 [16, 4, 0, 0]
1005 [12, 11, 19, 0]
1006 [23, 0, 9, 0]
1007 [6, 0, 0, 9]
1008 [0, 0, 0, 0]
1009 [0, 9,

In [38]:
###################################################################
#        Convolution 6 + ReLU
###################################################################
# Convolution
# - in:        (n, 256, 4, 4)
# - out:       (n, 256, 4, 4)
# - weight:  (256, 256, 3, 3)
# - bias:               (256)
# ReLU
# - in:        (n. 256. 4. 4)
# - out:       (n. 256. 4. 4)
###################################################################
I = {'IN_CH': 256, 'OUT_CH': 256, 'FLEN': 4}
F = {'BASE_ADDR': 0x0670_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0400_0000, 'STRIDE_SIZE': int(256*256*9/16), 'HSIZE': int(256*256*9/16), 'VSIZE': 16}
B = {'BASE_ADDR': 0x0490_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0680_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)

1

In [39]:
a = 0x0680_0000
for i in range(int(256*4*4/4)):
    temp = SU.su_read_data(a + 4*i)
    print(i, [temp[3],temp[2],temp[1],temp[0]])

0 [16, 19, 12, 5]
1 [5, 8, 9, 4]
2 [0, 7, 11, 8]
3 [2, 11, 7, 5]
4 [5, 4, 3, 1]
5 [6, 3, 0, 1]
6 [8, 7, 0, 1]
7 [6, 2, 0, 0]
8 [9, 6, 6, 4]
9 [2, 0, 0, 2]
10 [1, 0, 1, 2]
11 [1, 3, 4, 3]
12 [6, 11, 11, 17]
13 [5, 10, 11, 12]
14 [3, 10, 8, 8]
15 [6, 6, 6, 7]
16 [0, 0, 3, 0]
17 [0, 3, 10, 7]
18 [2, 7, 12, 15]
19 [2, 9, 10, 8]
20 [4, 7, 7, 5]
21 [5, 10, 13, 12]
22 [5, 10, 11, 9]
23 [2, 3, 4, 2]
24 [3, 2, 0, 2]
25 [1, 2, 2, 2]
26 [3, 3, 3, 2]
27 [3, 3, 2, 2]
28 [5, 8, 9, 4]
29 [5, 10, 5, 1]
30 [3, 0, 0, 1]
31 [2, 0, 0, 1]
32 [0, 0, 3, 0]
33 [0, 0, 1, 0]
34 [0, 0, 3, 1]
35 [0, 0, 0, 1]
36 [3, 4, 3, 2]
37 [4, 6, 6, 5]
38 [2, 3, 7, 6]
39 [1, 0, 4, 4]
40 [0, 2, 2, 0]
41 [1, 5, 3, 0]
42 [5, 7, 0, 0]
43 [2, 1, 0, 1]
44 [1, 5, 1, 5]
45 [2, 3, 5, 0]
46 [1, 0, 3, 2]
47 [3, 3, 6, 2]
48 [1, 3, 2, 3]
49 [8, 7, 5, 2]
50 [9, 8, 8, 2]
51 [7, 7, 5, 6]
52 [6, 6, 4, 4]
53 [5, 0, 0, 2]
54 [5, 1, 0, 5]
55 [4, 3, 8, 9]
56 [3, 3, 3, 7]
57 [6, 5, 13, 11]
58 [6, 5, 7, 8]
59 [6, 6, 5, 4]
60 [0, 0, 1, 1]
61 [3, 3, 

485 [3, 6, 0, 2]
486 [10, 11, 8, 5]
487 [15, 9, 5, 1]
488 [2, 5, 7, 4]
489 [2, 8, 8, 5]
490 [4, 5, 4, 8]
491 [7, 10, 12, 9]
492 [3, 4, 2, 0]
493 [3, 5, 1, 0]
494 [0, 0, 0, 0]
495 [0, 0, 0, 0]
496 [6, 5, 6, 7]
497 [6, 3, 13, 12]
498 [7, 5, 11, 10]
499 [7, 4, 6, 6]
500 [1, 0, 0, 0]
501 [3, 4, 0, 0]
502 [3, 7, 3, 0]
503 [1, 3, 2, 0]
504 [0, 0, 0, 0]
505 [4, 0, 0, 3]
506 [1, 0, 0, 2]
507 [2, 6, 5, 3]
508 [4, 5, 3, 2]
509 [5, 4, 3, 0]
510 [8, 4, 1, 1]
511 [4, 2, 2, 4]
512 [2, 3, 4, 3]
513 [3, 3, 8, 7]
514 [2, 3, 2, 5]
515 [3, 4, 3, 4]
516 [0, 4, 2, 0]
517 [4, 4, 5, 4]
518 [5, 5, 7, 7]
519 [5, 8, 6, 4]
520 [4, 5, 3, 1]
521 [3, 4, 0, 0]
522 [5, 13, 12, 8]
523 [9, 18, 16, 12]
524 [3, 1, 1, 3]
525 [3, 2, 5, 4]
526 [5, 4, 7, 5]
527 [5, 5, 6, 6]
528 [1, 3, 1, 1]
529 [0, 0, 0, 0]
530 [0, 0, 0, 0]
531 [0, 0, 1, 3]
532 [8, 10, 16, 12]
533 [5, 6, 12, 8]
534 [8, 12, 10, 9]
535 [5, 7, 10, 9]
536 [4, 0, 2, 0]
537 [3, 5, 11, 2]
538 [6, 7, 11, 7]
539 [6, 6, 6, 9]
540 [2, 6, 5, 0]
541 [4, 6, 4, 3]
542 [4, 

956 [2, 3, 5, 3]
957 [0, 1, 5, 4]
958 [3, 1, 0, 4]
959 [4, 2, 0, 4]
960 [3, 6, 7, 7]
961 [9, 20, 25, 14]
962 [2, 7, 11, 11]
963 [3, 6, 4, 3]
964 [3, 8, 11, 8]
965 [0, 1, 9, 7]
966 [0, 0, 1, 7]
967 [7, 5, 7, 5]
968 [2, 8, 17, 18]
969 [1, 12, 22, 16]
970 [4, 15, 21, 15]
971 [3, 9, 9, 9]
972 [3, 5, 3, 0]
973 [0, 0, 1, 0]
974 [1, 0, 0, 2]
975 [1, 6, 2, 4]
976 [4, 3, 4, 0]
977 [4, 4, 7, 0]
978 [4, 4, 1, 2]
979 [3, 2, 0, 1]
980 [7, 8, 6, 7]
981 [4, 7, 6, 11]
982 [11, 12, 13, 9]
983 [13, 8, 9, 5]
984 [0, 1, 2, 3]
985 [1, 0, 0, 2]
986 [0, 0, 0, 2]
987 [3, 1, 0, 0]
988 [0, 0, 0, 0]
989 [3, 0, 1, 1]
990 [0, 0, 6, 6]
991 [0, 4, 7, 5]
992 [2, 3, 2, 3]
993 [0, 2, 0, 2]
994 [0, 0, 2, 1]
995 [0, 3, 8, 7]
996 [5, 9, 6, 4]
997 [5, 7, 9, 10]
998 [5, 6, 7, 8]
999 [6, 5, 4, 7]
1000 [6, 5, 5, 5]
1001 [5, 0, 0, 0]
1002 [3, 0, 0, 1]
1003 [11, 10, 5, 8]
1004 [0, 0, 2, 1]
1005 [0, 0, 0, 0]
1006 [0, 0, 0, 1]
1007 [0, 0, 2, 1]
1008 [7, 9, 4, 3]
1009 [2, 4, 0, 1]
1010 [1, 4, 0, 0]
1011 [3, 6, 6, 2]
1012 [3, 0, 0,

In [40]:
###################################################################
#        Max Pool 4
###################################################################
# Max Pooling
# - in:      (n. 256. 4. 4)
# - out:     (n, 256, 2, 2)
###################################################################
I = {'IN_CH': 256, 'FLEN': 4}
F = {'BASE_ADDR': 0x0680_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
R = {'BASE_ADDR': 0x0690_0000, 'STRIDE_SIZE': 256*2*2, 'HSIZE': 256*2*2, 'VSIZE': 1}
SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)

1

In [42]:
a = 0x0690_0000
for i in range(int(256*2*2/4)):
    temp = SU.su_read_data(a + 4*i)
    print(i, [temp[3],temp[2]])
    print(i+1, [temp[1],temp[0]])

0 [19, 12]
1 [11, 11]
1 [6, 3]
2 [8, 1]
2 [9, 6]
3 [3, 4]
3 [11, 17]
4 [10, 8]
4 [3, 10]
5 [9, 15]
5 [10, 13]
6 [10, 11]
6 [3, 2]
7 [3, 3]
7 [10, 9]
8 [3, 1]
8 [0, 3]
9 [0, 3]
9 [6, 6]
10 [3, 7]
10 [5, 3]
11 [7, 1]
11 [5, 5]
12 [3, 6]
12 [8, 5]
13 [9, 8]
13 [6, 4]
14 [5, 9]
14 [6, 13]
15 [6, 8]
15 [3, 9]
16 [3, 2]
16 [0, 0]
17 [0, 0]
17 [2, 5]
18 [5, 7]
18 [11, 10]
19 [8, 10]
19 [6, 4]
20 [5, 6]
20 [3, 2]
21 [5, 1]
21 [6, 2]
22 [6, 2]
22 [3, 0]
23 [0, 0]
23 [9, 8]
24 [7, 8]
24 [2, 1]
25 [3, 7]
25 [8, 9]
26 [1, 2]
26 [4, 13]
27 [0, 0]
27 [11, 11]
28 [16, 16]
28 [13, 7]
29 [0, 1]
29 [7, 7]
30 [8, 21]
30 [3, 0]
31 [6, 4]
31 [10, 3]
32 [6, 8]
32 [1, 0]
33 [7, 9]
33 [11, 9]
34 [4, 10]
34 [6, 7]
35 [7, 8]
35 [4, 4]
36 [13, 8]
36 [15, 15]
37 [12, 9]
37 [1, 2]
38 [0, 2]
38 [4, 5]
39 [4, 4]
39 [6, 3]
40 [9, 7]
40 [2, 2]
41 [3, 0]
41 [6, 11]
42 [7, 3]
42 [13, 7]
43 [9, 12]
43 [13, 9]
44 [12, 10]
44 [7, 5]
45 [2, 4]
45 [10, 11]
46 [9, 2]
46 [18, 15]
47 [10, 8]
47 [11, 12]
48 [12, 6]
48 [6, 6]
49 

In [43]:
###################################################################
#        Fully-Connected 1 + ReLU
###################################################################
# Fully-Connected
# - in:             (1024,)
# - out:             (256,)
# - weight:     (256, 1024)
# - bias:            (256,)
# ReLU
# - in:              (256,)
# - out:             (256,)
###################################################################
F = {'BASE_ADDR': 0x0690_0000, 'STRIDE_SIZE': 1024, 'HSIZE': 1024, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0500_0000, 'STRIDE_SIZE': int(1024*256/8), 'HSIZE': int(1024*256/8), 'VSIZE': 8}
B = {'BASE_ADDR': 0x0530_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
R = {'BASE_ADDR': 0x06A0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)

1

In [51]:
a = 0x06A0_0000
for i in range(int(256/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end=' ')

[15, 23, 27, 0] [12, 7, 0, 0] [0, 26, 12, 35] [0, 0, 2, 9] [1, 3, 1, 0] [16, 24, 23, 4] [15, 31, 0, 11] [16, 21, 39, 17] [21, 7, 21, 0] [28, 1, 13, 21] [1, 27, 36, 27] [15, 16, 0, 9] [11, 17, 8, 15] [17, 19, 0, 4] [32, 0, 0, 9] [0, 1, 13, 1] [24, 10, 0, 13] [9, 0, 14, 24] [10, 19, 32, 14] [25, 0, 17, 10] [6, 0, 37, 0] [0, 1, 0, 4] [39, 0, 19, 0] [0, 0, 9, 9] [13, 8, 0, 18] [0, 14, 42, 32] [0, 15, 23, 39] [2, 34, 9, 6] [0, 4, 0, 15] [12, 26, 28, 0] [19, 14, 22, 0] [14, 0, 10, 24] [15, 21, 16, 0] [10, 4, 11, 26] [0, 0, 12, 2] [10, 3, 0, 0] [13, 14, 2, 10] [0, 3, 13, 18] [8, 6, 14, 0] [0, 4, 25, 4] [22, 0, 23, 0] [0, 17, 23, 46] [9, 12, 26, 3] [0, 0, 14, 0] [0, 16, 9, 37] [23, 29, 0, 1] [0, 14, 46, 0] [0, 23, 10, 22] [11, 32, 0, 0] [0, 0, 18, 0] [27, 16, 0, 4] [14, 0, 0, 4] [6, 45, 0, 14] [17, 3, 8, 9] [0, 13, 0, 2] [9, 21, 21, 26] [0, 7, 50, 10] [0, 16, 17, 0] [15, 30, 0, 6] [0, 14, 22, 3] [0, 0, 0, 0] [0, 18, 2, 0] [7, 0, 8, 29] [12, 16, 9, 34] 

In [45]:
###################################################################
#        Fully-Connected 2 + ReLU
###################################################################
# Fully-Connected
# - in:             (256,)
# - out:             (64,)
# - weight:      (64, 256)
# - bias:            (64,)
# ReLU
# - in:              (64,)
# - out:             (64,)
###################################################################
F = {'BASE_ADDR': 0x06A0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0540_0000, 'STRIDE_SIZE': 256*64, 'HSIZE': 256*64, 'VSIZE': 1}
B = {'BASE_ADDR': 0x0550_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
R = {'BASE_ADDR': 0x06B0_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)

1

In [52]:
a = 0x06B0_0000
for i in range(int(64/4)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end=' ')

[62, 31, 46, 0] [58, 70, 66, 0] [42, 4, 53, 67] [0, 0, 5, 0] [33, 75, 51, 53] [0, 0, 93, 127] [38, 35, 56, 0] [12, 0, 118, 25] [13, 0, 18, 30] [0, 39, 0, 45] [42, 17, 50, 80] [90, 122, 54, 28] [68, 0, 4, 19] [51, 0, 39, 62] [59, 46, 14, 70] [0, 0, 53, 43] 

In [47]:
###################################################################
#        Fully-Connected 3 + ReLU
###################################################################
# Fully-Connected
# - in:               (64,)
# - out:              (10,)
# - weight:        (10, 64)
# - bias:             (10,)
# ReLU
# - in:               (10,)
# - out:              (10,)
###################################################################
F = {'BASE_ADDR': 0x06B0_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
W = {'BASE_ADDR': 0x0560_0000, 'STRIDE_SIZE': 640, 'HSIZE': 640, 'VSIZE': 1}
B = {'BASE_ADDR': 0x0570_0000, 'STRIDE_SIZE': 10, 'HSIZE': 10, 'VSIZE': 1}
R = {'BASE_ADDR': 0x06C0_0000, 'STRIDE_SIZE': 10, 'HSIZE': 10, 'VSIZE': 1}
SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)

1

In [53]:
a = 0x06C0_0000
for i in range(int(3)):
    temp = SU.su_read_data(a + 4*i)
    print([temp[3],temp[2],temp[1],temp[0]], end=' ')

[27, 47, 168, 144] [228, 184, 44, 237] [127, 19, 15, 245] 

In [54]:
##############################################################################################
# Below code can be revised according to your apb register setting
##############################################################################################
# Read label index from apb register (our design output the index to that address)

# We assign FC_BASE_ADDR + 0x20 apb register to return max-value index
label = SU.su_read_data(FC_BASE_ADDR + 0x20)
label = int.from_bytes(label, 'big', signed=True)
# Predicted (computated) label
print(label-1)

7


In [55]:
# Real value
print(y_test[0])

3


### All Inference function

In [ ]:
def inference(image_idx):
    I = {'IN_CH': 3, 'OUT_CH': 32, 'FLEN': 32}
    F = {'BASE_ADDR': 0x0000_0000 + 3072*image_idx, 'STRIDE_SIZE': 3*32*32, 'HSIZE': 3*32*32, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0200_0000, 'STRIDE_SIZE': 3*32*9, 'HSIZE': 3*32*9, 'VSIZE': 1}
    B = {'BASE_ADDR': 0x0210_0000, 'STRIDE_SIZE': 32, 'HSIZE': 32, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0600_0000, 'STRIDE_SIZE': 32*32*32, 'HSIZE': 32*32*32, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 32, 'FLEN': 32}
    F = {'BASE_ADDR': 0x0600_0000, 'STRIDE_SIZE': 32*32*32, 'HSIZE': 32*32*32, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0610_0000, 'STRIDE_SIZE': 32*16*16, 'HSIZE': 32*16*16, 'VSIZE': 1}
    SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)
    I = {'IN_CH': 32, 'OUT_CH': 64, 'FLEN': 16}
    F = {'BASE_ADDR': 0x0610_0000, 'STRIDE_SIZE': 32*16*16, 'HSIZE': 32*16*16, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0220_0000, 'STRIDE_SIZE': 32*64*9, 'HSIZE': 32*64*9, 'VSIZE': 1}
    B = {'BASE_ADDR': 0x0270_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0620_0000, 'STRIDE_SIZE': 64*16*16, 'HSIZE': 64*16*16, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 64, 'FLEN': 16}
    F = {'BASE_ADDR': 0x0620_0000, 'STRIDE_SIZE': 64*16*16, 'HSIZE': 64*16*16, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0630_0000, 'STRIDE_SIZE': 64*8*8, 'HSIZE': 64*8*8, 'VSIZE': 1}
    SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)
    I = {'IN_CH': 64, 'OUT_CH': 128, 'FLEN': 8}
    F = {'BASE_ADDR': 0x0630_0000, 'STRIDE_SIZE': 64*8*8, 'HSIZE': 64*8*8, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0280_0000, 'STRIDE_SIZE': int(64*128*9/2), 'HSIZE': int(64*128*9/2), 'VSIZE': 2}
    B = {'BASE_ADDR': 0x02C0_0000, 'STRIDE_SIZE': 128, 'HSIZE': 128, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0640_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 128, 'OUT_CH': 128, 'FLEN': 8}
    F = {'BASE_ADDR': 0x0640_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0300_0000, 'STRIDE_SIZE': int(128*128*9/4), 'HSIZE': int(128*128*9/4), 'VSIZE': 4}
    B = {'BASE_ADDR': 0x0390_0000, 'STRIDE_SIZE': 128, 'HSIZE': 128, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0650_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 128, 'FLEN': 8}
    F = {'BASE_ADDR': 0x0650_0000, 'STRIDE_SIZE': 128*8*8, 'HSIZE': 128*8*8, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0660_0000, 'STRIDE_SIZE': 128*4*4, 'HSIZE': 128*4*4, 'VSIZE': 1}
    SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)
    I = {'IN_CH': 128, 'OUT_CH': 256, 'FLEN': 4}
    F = {'BASE_ADDR': 0x0660_0000, 'STRIDE_SIZE': 128*4*4, 'HSIZE': 128*4*4, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x03A0_0000, 'STRIDE_SIZE': int(128*256*9/8), 'HSIZE': int(128*256*9/8), 'VSIZE': 8}
    B = {'BASE_ADDR': 0x03F0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0670_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 256, 'OUT_CH': 256, 'FLEN': 4}
    F = {'BASE_ADDR': 0x0670_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0400_0000, 'STRIDE_SIZE': int(256*256*9/16), 'HSIZE': int(256*256*9/16), 'VSIZE': 16}
    B = {'BASE_ADDR': 0x0490_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0680_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
    SU.su_conv_control(I, F, W, B, R, VDMA1_BASE_ADDR, CONV_BASE_ADDR)
    I = {'IN_CH': 256, 'FLEN': 4}
    F = {'BASE_ADDR': 0x0680_0000, 'STRIDE_SIZE': 256*4*4, 'HSIZE': 256*4*4, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x0690_0000, 'STRIDE_SIZE': 256*2*2, 'HSIZE': 256*2*2, 'VSIZE': 1}
    SU.su_pool_control(I, F, R, VDMA2_BASE_ADDR, POOL_BASE_ADDR)
    F = {'BASE_ADDR': 0x0690_0000, 'STRIDE_SIZE': 1024, 'HSIZE': 1024, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0500_0000, 'STRIDE_SIZE': int(1024*256/8), 'HSIZE': int(1024*256/8), 'VSIZE': 8}
    B = {'BASE_ADDR': 0x0530_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x06A0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
    SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)
    F = {'BASE_ADDR': 0x06A0_0000, 'STRIDE_SIZE': 256, 'HSIZE': 256, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0540_0000, 'STRIDE_SIZE': 256*64, 'HSIZE': 256*64, 'VSIZE': 1}
    B = {'BASE_ADDR': 0x0550_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x06B0_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
    SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)
    F = {'BASE_ADDR': 0x06B0_0000, 'STRIDE_SIZE': 64, 'HSIZE': 64, 'VSIZE': 1}
    W = {'BASE_ADDR': 0x0560_0000, 'STRIDE_SIZE': 640, 'HSIZE': 640, 'VSIZE': 1}
    B = {'BASE_ADDR': 0x0570_0000, 'STRIDE_SIZE': 10, 'HSIZE': 10, 'VSIZE': 1}
    R = {'BASE_ADDR': 0x06C0_0000, 'STRIDE_SIZE': 10, 'HSIZE': 10, 'VSIZE': 1}
    SU.su_fc_control(F, W, B, R, VDMA0_BASE_ADDR, FC_BASE_ADDR)
    ##############################################################################################
    # Below code can be revised according to your apb register setting
    ##############################################################################################
    label = SU.su_read_data(FC_BASE_ADDR + 0x20)
    label = int.from_bytes(label, 'big', signed=True)
    # print(label-1)
    return (label)

### Check accuracy

In [ ]:
acc = 0
for i in range(20):
    pred = inference(i)
    if pred == y_test[i]:
        acc += 1
    print("Progress: {:05.2f}%".format(100*i/100), end="\r", flush=True)
    # show sample image and predict result
    if i % 1 == 0:
        gen_image(X_test_origin[i]).show()
        print("Label: %d (%s)" %(y_test[i], label_list[y_test[i]]))
        print("Predict: %d (%s)" %(pred, label_list[pred]))
print("\t100 images accuracy: {:.2f}%".format(acc/100 * 100))

In [ ]:
x = SU.su_read_data(FC_BASE_ADDR + 0x30)
y = SU.su_read_data(FC_BASE_ADDR + 0x34)
z = SU.su_read_data(FC_BASE_ADDR + 0x3c)
x = int.from_bytes(x, 'big', signed=True)
y = int.from_bytes(y, 'big', signed=True)
z = int.from_bytes(z, 'big', signed=True)
print(x)
print(y)
print(z)

xa = SU.su_read_data(FC_BASE_ADDR + 0x40)
ya = SU.su_read_data(FC_BASE_ADDR + 0x44)
za = SU.su_read_data(FC_BASE_ADDR + 0x4c)
print(xa)
print(ya)
print(za)
xa = int.from_bytes(xa, 'big', signed=True)
ya = int.from_bytes(ya, 'big', signed=True)
za = int.from_bytes(za, 'big', signed=True)
print(xa)
print(ya)
print(za)